## Initial Data Loading and Preparation

This section consolidates all steps required to load raw CSV files, normalize their columns, and merge them into a single `merged_df` DataFrame, ensuring all subsequent analyses have a consistent and ready-to-use dataset.

### Mount Google Drive

Mount your Google Drive to access the CSV files.

In [ ]:
!pip install -r ../requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements'


In [ ]:
import os
import pandas as pd
import unicodedata
from pathlib import Path

def load_all_csv_from_data_folder(directory_path='../data'):
    dataframes = {}
    path = Path(directory_path)
    if not path.exists():
        print(f"Error: El directorio '{directory_path}' no existe.")
        return dataframes

    print(f"Buscando ficheros CSV en: {directory_path} (recursivo)")
    exclude_files = {'dataset_final', 'secciones_area_censo2021', 'distancias_servicios'}
    for file_path in path.rglob('*.csv'):
        if 'venv' in file_path.parts or 'site-packages' in file_path.parts:
            continue
        if file_path.stem in exclude_files:
            continue
        filename = file_path.name
        df_name = file_path.stem
        try:
            dataframes[df_name] = pd.read_csv(file_path)
            print(f"  -> Cargado '{filename}' desde '{file_path.parent}' como '{df_name}'.")
        except Exception as e:
            print(f"  -> Error al cargar '{filename}': {e}")
    if not dataframes:
        print("No se encontraron ficheros CSV en el directorio especificado.")
    return dataframes

def clean_column_name(col_name):
    col_name = str(col_name).lower()
    col_name = ''.join(c for c in col_name if c.isalnum() or c.isspace())
    col_name = unicodedata.normalize('NFKD', col_name).encode('ascii', 'ignore').decode('utf-8')
    col_name = col_name.replace(' ', '_')
    col_name = ''.join(c for c in col_name if c.isalnum() or c == '_')
    col_name = '_'.join(filter(None, col_name.split('_')))
    return col_name

### Load Raw DataFrames

Specify the path to your CSV files in Google Drive and load them.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local execution: Google Drive mount skipped.')

google_drive_csv_folder_path = "../data"
print(f"Cargando ficheros CSV desde: {google_drive_csv_folder_path}")
ground_truth_dataframes = load_all_csv_from_data_folder(google_drive_csv_folder_path)

if ground_truth_dataframes:
    print("\nSe han cargado los siguientes DataFrames:")
    for name, df in ground_truth_dataframes.items():
        print(f"- '{name}' (Filas: {df.shape[0]}, Columnas: {df.shape[1]})")
else:
    print("No se cargó ningún DataFrame.")

### Normalize Column Names

Applying the `clean_column_name` function to all loaded DataFrames.

In [ ]:
normalized_dataframes = {}
print("Normalizando nombres de columnas para todos los DataFrames...")
for df_name, df in ground_truth_dataframes.items():
    original_columns = df.columns.tolist()
    df.columns = [clean_column_name(col) for col in df.columns]
    normalized_dataframes[df_name] = df.copy() # Almacenar una copia para evitar SettingWithCopyWarning
    print(f"  -> DataFrame '{df_name}':")
    print(f"     Columnas originales: {original_columns}")
    print(f"     Columnas normalizadas: {df.columns.tolist()}")

print("\nNormalización completada. Aquí están los primeros DataFrames con sus columnas normalizadas:")
for name, df in list(normalized_dataframes.items())[:2]: # Mostrar los primeros 2 como ejemplo
    print(f"\nDataFrame '{name}':")
    display(df.head())

### Merge All DataFrames into `merged_df`

Merging all normalized dataframes into a single `merged_df` and saving it locally.

In [ ]:
from functools import reduce
import pandas as pd
import os

# Asegurarse de que normalized_dataframes esté disponible
if 'normalized_dataframes' not in locals():
    print("Error: 'normalized_dataframes' no se encuentra. Asegúrate de haber ejecutado las celdas anteriores.")
else:
    # Definir patrones de nombres para columnas de metadatos (solo se aplicarán a columnas de tipo 'object')
    metadata_name_patterns = [
        'nota', 'serie', 'unidad', 'indicador', 'dato',
        'idresidencian', 'objectid', 'cnut', 'cca', 'codigo',
        'clau', 'cumun', 'cpro', 'cdis', 'cudis', 'anyo'
    ]

    # Columnas que siempre se deben mantener, incluso si coinciden con un patrón
    # 'cusec' es la clave de unión, 'isla' y 'nmun' son cruciales para la imputación jerárquica.
    columns_to_explicitly_keep = ['cusec', 'isla', 'nmun']

    # Crear un nuevo diccionario para almacenar DataFrames sin metadatos
    filtered_dataframes = {}
    print("Filtrando columnas de metadatos (tipo 'object') de los DataFrames antes del merge...")

    # --- FIX START: Exclude 'merged_ground_truth_data' from processing ---
    dataframes_to_process = normalized_dataframes.copy()
    if 'merged_ground_truth_data' in dataframes_to_process:
        print("Excluyendo 'merged_ground_truth_data' de la lista de DataFrames a procesar, ya que es el resultado del merge.")
        del dataframes_to_process['merged_ground_truth_data']

    for df_name, df in dataframes_to_process.items(): # Iterate over the filtered dictionary
        cols_to_drop_from_df = []
        for col in df.columns:
            # Solo considerar columnas de tipo 'object' para esta limpieza
            # Y asegurar que no sean columnas explícitamente a mantener
            if df[col].dtype == 'object' and col not in columns_to_explicitly_keep:
                # Check if column name matches any metadata pattern
                if any(pattern in col for pattern in metadata_name_patterns):
                    cols_to_drop_from_df.append(col)

        if cols_to_drop_from_df:
            print(f"  -> Eliminando {len(cols_to_drop_from_df)} columnas de metadatos (tipo 'object') de '{df_name}': {cols_to_drop_from_df}")
            filtered_dataframes[df_name] = df.drop(columns=cols_to_drop_from_df, errors='ignore').copy()
        else:
            filtered_dataframes[df_name] = df.copy()
            print(f"  -> No se encontraron columnas de metadatos (tipo 'object') para eliminar en '{df_name}'.")

    # Extraer el primer DataFrame para iniciar el merge de los DataFrames filtrados
    all_df_items = list(filtered_dataframes.items())

    if not all_df_items:
        print("No hay DataFrames en 'filtered_dataframes' para unir después de la limpieza.")
    else:
        first_df_name, merged_df = all_df_items[0]
        merged_df = merged_df.copy()
        print(f"Iniciando el merge con '{first_df_name}' (shape: {merged_df.shape}).\n")

        # Define columns that should ideally appear only once (identifiers, metadata).
        # These will be dropped from the right DataFrame if they already exist in the left.
        cols_to_deduplicate = [
            'isla', 'nmun', 'cca', 'cnut0', 'cnut1', 'cnut2', 'cnut3', 'csec', 'codigo',
            'idregion', 'cmun', 'anyo', 'cumun', 'cpro', 'npro', 'nca', 'cdis', 'cudis',
            'objectid', 'objectid1', 'lon', 'lat', 'clau2',
            'shapelength', 'shapeleng', 'shapearea', # Assuming these are descriptive geometries of the cusec
        ]

        # Iterar sobre el resto de DataFrames y unirlos
        for df_name, df_to_merge_original in all_df_items[1:]:
            df_to_merge = df_to_merge_original.copy() # Work on a copy

            if 'cusec' in df_to_merge.columns:
                # Identify columns in df_to_merge that are also in merged_df
                # and are in our deduplication list (excluding 'cusec').
                cols_to_drop_from_right = [
                    col for col in cols_to_deduplicate
                    if col in df_to_merge.columns and col in merged_df.columns
                ]

                if cols_to_drop_from_right:
                    print(f"  -> Eliminando columnas de ID/metadata duplicadas de '{df_name}' antes del merge: {cols_to_drop_from_right}")
                    df_to_merge = df_to_merge.drop(columns=cols_to_drop_from_right, errors='ignore')

                # Now merge. No need for `suffixes` if we've proactively dropped duplicates from the right.
                print(f"Uniendo '{df_name}' (shape: {df_to_merge_original.shape} -> filtered shape: {df_to_merge.shape}) con el DataFrame consolidado.")
                merged_df = pd.merge(merged_df, df_to_merge, on='cusec', how='left')

                print(f"  -> Nuevo shape del DataFrame consolidado: {merged_df.shape}\n")
            else:
                print(f"Advertencia: El DataFrame '{df_name}' no tiene la columna 'cusec'. Se omitirá en el merge.\n")

        cartografia = pd.read_csv('../data/dataset_canarias_raw.csv')
        isla_lookup = cartografia.set_index('CUSEC')['isla'].to_dict()
        merged_df['isla'] = merged_df['cusec'].map(isla_lookup) # FIXED: Removed .astype(str) here

        print("\n--- Merge Finalizado ---")
        print("Información del DataFrame unido (merged_df):")
        merged_df.info()
        display(merged_df.head())

### Aplicar Normalización a Todos los DataFrames

Ahora aplicaremos esta función a las columnas de todos los DataFrames que cargamos desde Google Drive. Los DataFrames normalizados se almacenarán en un nuevo diccionario llamado `normalized_dataframes`.

### Validación de Rango y Consistencia de Variables

Realizaremos las validaciones solicitadas para detectar posibles errores o inconsistencias en los datos.

In [ ]:
print("\n--- Realizando validación de datos ---")

# 1. pct_extranjeros debe estar en [0, 100]
if 'poblacion_canarias' in normalized_dataframes:
    df_poblacion = normalized_dataframes['poblacion_canarias']
    if 'pctextranjeros' in df_poblacion.columns:
        invalid_pct_extranjeros = df_poblacion[(df_poblacion['pctextranjeros'] < 0) | (df_poblacion['pctextranjeros'] > 100)]
        if not invalid_pct_extranjeros.empty:
            print(f"\nAdvertencia en 'poblacion_canarias': {len(invalid_pct_extranjeros)} filas tienen 'pctextranjeros' fuera del rango [0, 100].")
            display(invalid_pct_extranjeros[['cusec', 'pctextranjeros']].head())
        else:
            print("\n'poblacion_canarias': 'pctextranjeros' está dentro del rango [0, 100].")

# 2. pct_menor_18 + pct_mayor_65 no puede superar 100
if 'indice_gini_canarias' in normalized_dataframes:
    df_gini = normalized_dataframes['indice_gini_canarias']
    if 'pctmenor18' in df_gini.columns and 'pctmayor65' in df_gini.columns:
        invalid_sum_pct = df_gini[(df_gini['pctmenor18'] + df_gini['pctmayor65']) > 100]
        if not invalid_sum_pct.empty:
            print(f"\nAdvertencia en 'indice_gini_canarias': {len(invalid_sum_pct)} filas tienen 'pctmenor18' + 'pctmayor65' > 100.")
            display(invalid_sum_pct[['cusec', 'pctmenor18', 'pctmayor65']].head())
        else:
            print("\n'indice_gini_canarias': 'pctmenor18' + 'pctmayor65' no supera 100.")

# 3. distminhospitalkm debe ser > 0
for df_name in ['dataset_canarias_raw', 'distancias_servicios']:
    if df_name in normalized_dataframes:
        df = normalized_dataframes[df_name]
        if 'distminhospitalkm' in df.columns:
            invalid_dist_hospital = df[df['distminhospitalkm'] <= 0]
            if not invalid_dist_hospital.empty:
                print(f"\nAdvertencia en '{df_name}': {len(invalid_dist_hospital)} filas tienen 'distminhospitalkm' <= 0.")
                display(invalid_dist_hospital[['cusec', 'distminhospitalkm']].head())
            else:
                print(f"\n'{df_name}': 'distminhospitalkm' es > 0.")

# 4. area_km2 debe ser > 0 (asumiendo 'shapearea')
shapearea_dfs = ['salario_renta_canarias', 'indice_gini_canarias', 'pensiones_renta_canarias', 'p80p20_canarias', 'renta_media_hogar_canarias']
for df_name in shapearea_dfs:
    if df_name in normalized_dataframes:
        df = normalized_dataframes[df_name]
        if 'shapearea' in df.columns:
            invalid_area = df[df['shapearea'] <= 0]
            if not invalid_area.empty:
                print(f"\nAdvertencia en '{df_name}': {len(invalid_area)} filas tienen 'shapearea' <= 0. Posible geometría rota.")
                display(invalid_area[['cusec', 'shapearea']].head())
            else:
                print(f"\n'{df_name}': 'shapearea' es > 0.")

# 5. densidad_real_hab_km2 = poblacion_total / area_km2
print("\n--- Validación de Densidad (Requiere uniones) ---")
print("La validación 'densidad_real_hab_km2 = poblacion_total / area_km2' requiere unir el DataFrame 'poblacion_canarias' (que contiene 'poblaciontotal' y 'densidadhabkm2') con los DataFrames que contienen 'shapearea' (como 'salario_renta_canarias' o 'indice_gini_canarias') usando una clave común (probablemente 'cusec').")
print("Una vez unidos, se podría calcular una 'densidad_calculada' y compararla con la 'densidadhabkm2' existente.")
print("\nSi deseas realizar esta unión y comparación, por favor, indícalo.")

print("\n--- Fin de la validación de datos ---")

### Unión de todos los DataFrames

Vamos a unir todos los DataFrames en `normalized_dataframes` en un único DataFrame grande, usando `cusec` como clave. Utilizaremos una unión `outer` para asegurarnos de que no se pierdan datos y manejaremos los nombres de columnas duplicados con sufijos.

Ahora que tenemos un DataFrame unificado, podemos continuar con las validaciones pendientes, como la de la densidad, y decidir cómo manejar los valores anómalos de `distminhospitalkm`.

### Validación Post-Join del DataFrame Consolidado (`merged_df`)

Ahora que hemos unido todos los DataFrames, es crucial realizar una validación para entender el impacto del proceso de unión y detectar posibles anomalías como Nulos, duplicados o CUSECs 'perdidos'.

In [ ]:
# 1. Comprobar Nulos introducidos por el merge
print("\n--- Análisis de Valores Nulos --- ")
initial_nan_counts = {df_name: df.isnull().sum().sum() for df_name, df in normalized_dataframes.items()}
print("Valores nulos iniciales por DataFrame (antes del merge):")
for name, count in initial_nan_counts.items():
    if count > 0:
        print(f"  - {name}: {count} nulos")

merged_nan_counts = merged_df.isnull().sum()
total_merged_nans = merged_nan_counts.sum()

print(f"\nTotal de valores nulos en el merged_df: {total_merged_nans}")

nan_columns = merged_nan_counts[merged_nan_counts > 0]
if not nan_columns.empty:
    print("Columnas con valores nulos (y su recuento):")
    display(nan_columns.sort_values(ascending=False).head(20))
else:
    print("No hay columnas con valores nulos en el merged_df.")

# 2. Comprobar duplicados en 'cusec'
print("\n--- Análisis de Duplicados --- ")
duplicated_cusec = merged_df[merged_df.duplicated(subset=['cusec'], keep=False)]
if not duplicated_cusec.empty:
    print(f"Advertencia: Se encontraron {len(duplicated_cusec)} filas con CUSEC duplicados.\n")
    display(duplicated_cusec.sort_values('cusec'))
else:
    print("No se encontraron CUSEC duplicados en el merged_df.")

# 3. Comparar CUSECs con los DataFrames originales
print("\n--- Comparación de CUSECs --- ")
merged_cusecs = set(merged_df['cusec'].dropna().unique())

# CUSECs en DataFrames originales
original_cusecs_sets = {}
for df_name, df in normalized_dataframes.items():
    if 'cusec' in df.columns:
        original_cusecs_sets[df_name] = set(df['cusec'].dropna().unique())

print(f"Número de CUSECs únicos en merged_df: {len(merged_cusecs)}")

all_original_cusecs = set()
for df_name, cusec_set in original_cusecs_sets.items():
    all_original_cusecs.update(cusec_set)

print(f"Número total de CUSECs únicos combinados de los DataFrames originales: {len(all_original_cusecs)}")

# CUSECs presentes en el merge que no estaban en ningún original (si `outer` join crea nuevos, aunque es raro con `cusec`)
newly_added_cusecs = merged_cusecs - all_original_cusecs
if newly_added_cusecs:
    print(f"\nAdvertencia: Se encontraron {len(newly_added_cusecs)} CUSECs en merged_df que no estaban en ningún DataFrame original. Esto podría indicar un problema en el origen de los datos o en la clave de unión.\n")
    print(list(newly_added_cusecs)[:10]) # Mostrar algunos ejemplos
else:
    print("No se encontraron CUSECs en merged_df que no estuvieran en los originales.")

# CUSECs originales que NO están en el merged_df (debido a NaNs en cusec o un merge mal configurado, poco probable con outer)
lost_cusecs = all_original_cusecs - merged_cusecs
if lost_cusecs:
    print(f"\nAdvertencia: Se perdieron {len(lost_cusecs)} CUSECs de los DataFrames originales en el merged_df. Estos CUSECs no aparecen en la columna 'cusec' del resultado final.\n")
    print(list(lost_cusecs)[:10]) # Mostrar algunos ejemplos
else:
    print("No se perdieron CUSECs de los DataFrames originales en el merged_df.")

print("\n--- Fin de la Validación Post-Join --- ")


El análisis ha detectado Nulos significativos, lo cual es esperable con una unión `outer` cuando DataFrames tienen diferentes conjuntos de CUSECs. La ausencia de duplicados en `cusec` es una buena señal.

Ahora, volvamos a las validaciones de consistencia de los datos, especialmente la de la densidad y el manejo de los `distminhospitalkm <= 0`.

### Análisis de Tipos y Estructura del Dataset Completo (`merged_df`)

Ahora vamos a examinar la estructura general del `merged_df`, incluyendo los tipos de datos de sus columnas, la cantidad de valores no nulos y un resumen estadístico.

In [ ]:
print("--- Información General del merged_df ---")
merged_df.info()

print("\n--- Resumen Estadístico de Columnas Numéricas ---")
display(merged_df.describe())

print("\n--- Resumen Estadístico de Columnas Categóricas/Objeto ---")
display(merged_df.describe(include='object'))

print("\n--- Conteo de Valores Únicos por Columna (primeras 20 con más nulos o baja cardinalidad) ---")
# Identificar columnas con nulos y/o baja cardinalidad para un mejor análisis
column_info = pd.DataFrame({
    'dtype': merged_df.dtypes,
    'non_null_count': merged_df.count(),
    'null_count': merged_df.isnull().sum(),
    'unique_count': merged_df.nunique()
})

# Ordenar por cantidad de nulos descendente, luego por cardinalidad ascendente
display(column_info.sort_values(by=['null_count', 'unique_count'], ascending=[False, True]).head(20))

Este análisis nos da una idea clara de la mezcla de tipos de datos, la prevalencia de valores nulos y la cardinalidad de las columnas, lo cual es fundamental para decidir los próximos pasos en la limpieza y preparación de los datos. Ahora que tenemos esta visión, podemos abordar las validaciones de consistencia de la densidad y el `distminhospitalkm`.

In [ ]:
print("--- Manejo de anomalías en 'distminhospitalkm' ---")

# Una distancia mínima a un hospital debe ser estrictamente positiva.
# Se corrigen únicamente los valores <= 0 y se conserva el resto de la variable.
hospital_col = 'distminhospitalkm'

if hospital_col not in merged_df.columns:
    raise KeyError(f"No se encuentra la columna '{hospital_col}' en merged_df.")

anomaly_mask = merged_df[hospital_col].notna() & (merged_df[hospital_col] <= 0)
anomalous_rows_count = int(anomaly_mask.sum())

if anomalous_rows_count > 0:
    print(f"Se encontraron {anomalous_rows_count} filas con '{hospital_col}' <= 0.")

    # Guardar el valor original y marcar las filas corregidas para trazabilidad.
    merged_df[hospital_col + '_original'] = merged_df[hospital_col]
    merged_df[hospital_col + '_imputed'] = False
    merged_df.loc[anomaly_mask, hospital_col + '_imputed'] = True

    # Nivel 1: mediana del municipio.
    municipality_median = merged_df.groupby('nmun')[hospital_col].transform(
        lambda s: s.where(s > 0).median()
    )
    use_municipality = anomaly_mask & municipality_median.notna()
    merged_df.loc[use_municipality, hospital_col] = municipality_median[use_municipality]

    # Nivel 2: mediana de la isla para los casos que no pudieron resolverse.
    remaining_mask = anomaly_mask & (merged_df[hospital_col] <= 0)
    island_median = merged_df.groupby('isla')[hospital_col].transform(
        lambda s: s.where(s > 0).median()
    )
    use_island = remaining_mask & island_median.notna()
    merged_df.loc[use_island, hospital_col] = island_median[use_island]

    # Nivel 3: mediana global positiva como último recurso.
    remaining_mask = anomaly_mask & (merged_df[hospital_col] <= 0)
    global_positive_median = merged_df.loc[
        merged_df[hospital_col] > 0, hospital_col
    ].median()

    # Corrección: usar pd.notna() en lugar de .notna() en un float
    import pandas as pd # Asegurarse de que pandas está importado para pd.notna
    use_global = remaining_mask & pd.notna(global_positive_median)
    merged_df.loc[use_global, hospital_col] = global_positive_median

    print(f"  -> Corregidos con mediana municipal: {int(use_municipality.sum())}")
    print(f"  -> Corregidos con mediana insular:   {int(use_island.sum())}")
    print(f"  -> Corregidos con mediana global:    {int(use_global.sum())}")

    unresolved = anomaly_mask & (merged_df[hospital_col] <= 0)
    if unresolved.any():
        print("Advertencia: quedan anomalías sin resolver.")
        display(merged_df.loc[unresolved, ['cusec', 'nmun', 'isla', hospital_col]])
    else:
        print("Verificación exitosa: todos los valores corregidos son > 0.")
else:
    print(f"No se encontraron anomalías en '{hospital_col}' (valores <= 0).")

In [ ]:
print("--- Trazabilidad de la corrección de 'distminhospitalkm' ---")

hospital_col = 'distminhospitalkm'
flag_col = hospital_col + '_imputed'
original_col = hospital_col + '_original'

if flag_col in merged_df.columns:
    affected_rows = merged_df.loc[
        merged_df[flag_col],
        ['cusec', 'isla', 'nmun', original_col, hospital_col]
    ].copy()

    print(f"Secciones corregidas: {len(affected_rows)}")
    if not affected_rows.empty:
        display(affected_rows)
else:
    print("No se ha creado la marca de trazabilidad; ejecuta primero la celda anterior.")

### Imputación Jerárquica de Valores Nulos

Procedemos a imputar los valores nulos en las columnas numéricas siguiendo la lógica:
1.  **Media del Municipio**: Se imputan los nulos con la media de su municipio.
2.  **Mediana de la Isla**: Si aún quedan nulos (porque todo el municipio era nulo), se imputan con la mediana de la isla.
3.  **Mediana Global**: Si todavía quedan nulos (casos muy raros donde toda la isla o las columnas `isla`/`nmun` eran nulas), se imputan con la mediana global de la columna.

Solo se considerarán para imputación columnas numéricas que tengan al menos 100 valores no nulos (para asegurar que las estadísticas de grupo sean representativas).

In [ ]:
import numpy as np

# Asegurarse de que merged_df existe (asumiendo que se ha creado en celdas anteriores)
if 'merged_df' not in locals():
    print("Error: 'merged_df' no se encuentra. Asegúrate de haber ejecutado las celdas anteriores.")
    # Puedes añadir aquí código para cargar merged_df si es necesario, ej: merged_df = pd.read_csv('data/merged_ground_truth_data.csv')
else:
    print("### Imputación de Valores Nulos con Estrategia Jerárquica ###\n")

    # Identificar columnas numéricas con valores nulos y con al menos 100 valores no nulos para intentar imputación.
    # Excluiremos 'cusec' ya que es el identificador y no debería ser imputado.
    numerical_cols_with_nan = merged_df.select_dtypes(include=['float64', 'int64']).columns[merged_df.select_dtypes(include=['float64', 'int64']).isnull().any()].tolist()
    imputable_cols = [col for col in numerical_cols_with_nan if merged_df[col].count() > 100 and col != 'cusec']

    print(f"Columnas numéricas candidatas para imputación (más de 100 valores no nulos y con NaNs): {len(imputable_cols)} columnas.")
    print(imputable_cols)

    if not imputable_cols:
        print("\nNo se encontraron columnas adecuadas para la imputación según los criterios definidos.")
    else:
        # Asegurarse de que 'isla' y 'nmun' son columnas para agrupar.
        # Si 'isla' o 'nmun' tienen nulos, esas filas no serán imputadas por grupo, sino por mediana global.
        if 'isla' not in merged_df.columns or 'nmun' not in merged_df.columns:
            print("Advertencia: Las columnas 'isla' o 'nmun' no se encontraron en merged_df. La imputación jerárquica por grupo puede no ser posible o completa.")
        else:
            # Pre-calcular nulos iniciales para el reporte
            original_nan_counts_report = merged_df[imputable_cols].isnull().sum()
            original_nan_counts_report = original_nan_counts_report[original_nan_counts_report > 0]
            if not original_nan_counts_report.empty:
                print("\nValores nulos iniciales en columnas a imputar:")
                print(original_nan_counts_report)
            else:
                print("\nNo hay valores nulos en las columnas candidatas antes de la imputación.")

            imputation_log = [] # Para documentar qué se imputó y cómo

            for col in imputable_cols:
                nan_before_imputation = merged_df[col].isnull().sum()
                if nan_before_imputation == 0:
                    continue # Ya no hay NaNs en esta columna, pasar a la siguiente

                print(f"\n--- Imputando columna: '{col}' (NaNs iniciales: {nan_before_imputation}) ---")

                # Step 1: Impute by municipality mean
                # Create a temporary Series with municipality means, aligned by index
                municipality_means = merged_df.groupby('nmun')[col].transform('mean')

                # Record cusecs imputed by municipality mean
                imputed_by_mun_mask = merged_df[col].isnull() & municipality_means.notna()
                imputed_cusecs_mun = merged_df.loc[imputed_by_mun_mask, 'cusec'].tolist()

                merged_df[col] = merged_df[col].fillna(municipality_means)
                nan_after_municipality_imputation = merged_df[col].isnull().sum()

                if len(imputed_cusecs_mun) > 0:
                    imputation_log.append({
                        'column': col,
                        'method': 'municipality_mean',
                        'count': len(imputed_cusecs_mun),
                        'cusecs_affected': imputed_cusecs_mun
                    })
                print(f"  -> Nulos restantes después de media municipal: {nan_after_municipality_imputation}")

                # Step 2: Impute by island median (for remaining NaNs)
                if nan_after_municipality_imputation > 0:
                    # Create a temporary Series with island medians, aligned by index
                    island_medians = merged_df.groupby('isla')[col].transform('median')

                    # Record cusecs imputed by island median
                    imputed_by_isla_mask = merged_df[col].isnull() & island_medians.notna()
                    imputed_cusecs_isla = merged_df.loc[imputed_by_isla_mask, 'cusec'].tolist()

                    merged_df[col] = merged_df[col].fillna(island_medians)
                    nan_after_island_imputation = merged_df[col].isnull().sum()

                    if len(imputed_cusecs_isla) > 0:
                        imputation_log.append({
                            'column': col,
                            'method': 'island_median',
                            'count': len(imputed_cusecs_isla),
                            'cusecs_affected': imputed_cusecs_isla
                        })
                    print(f"  -> Nulos restantes después de mediana de isla: {nan_after_island_imputation}")

                # Step 3: Impute by global median (for any last remaining NaNs)
                if nan_after_island_imputation > 0:
                    global_median = merged_df[col].median()

                    # Record cusecs imputed by global median
                    imputed_by_global_mask = merged_df[col].isnull()
                    imputed_cusecs_global = merged_df.loc[imputed_by_global_mask, 'cusec'].tolist()

                    merged_df[col] = merged_df[col].fillna(global_median)
                    nan_after_global_imputation = merged_df[col].isnull().sum()

                    if len(imputed_cusecs_global) > 0:
                        imputation_log.append({
                            'column': col,
                            'method': 'global_median',
                            'count': len(imputed_cusecs_global),
                            'cusecs_affected': imputed_cusecs_global
                        })
                    print(f"  -> Nulos restantes después de mediana global: {nan_after_global_imputation}")

            print("\n### Resumen Final de Imputación ###")
            if imputation_log:
                # Consolidar información para el reporte
                imputed_summary = pd.DataFrame(imputation_log)
                print("Detalle de imputaciones realizadas:")
                display(imputed_summary.groupby(['column', 'method']).agg(
                    total_imputed=('count', 'sum'),
                    example_cusecs=('cusecs_affected', lambda x: x.explode().unique()[:5].tolist()) # Mostrar hasta 5 CUSECs de ejemplo
                ))

                # Documentar qué secciones (cusecs) se vieron afectadas y en qué islas/municipios
                print("\n### Detalle por Secciones y Ubicación ###")
                affected_sections_full_detail = {} # {isla: {municipio: {cusec: {column: method}}}}'

                for entry in imputation_log:
                    column = entry['column']
                    method = entry['method']
                    cusecs = entry['cusecs_affected']

                    for cusec_val in cusecs:
                        # Recuperar la fila usando cusec, manejar posibles NaNs en 'isla' o 'nmun' para las columnas de agrupación
                        row = merged_df[merged_df['cusec'] == cusec_val].iloc[0]
                        isla_val = row['isla'] if pd.notna(row['isla']) else 'Desconocida'
                        nmun_val = row['nmun'] if pd.notna(row['nmun']) else 'Desconocido'

                        if isla_val not in affected_sections_full_detail:
                            affected_sections_full_detail[isla_val] = {}
                        if nmun_val not in affected_sections_full_detail[isla_val]:
                            affected_sections_full_detail[isla_val][nmun_val] = {}
                        if cusec_val not in affected_sections_full_detail[isla_val][nmun_val]:
                            affected_sections_full_detail[isla_val][nmun_val][cusec_val] = {}
                        affected_sections_full_detail[isla_val][nmun_val][cusec_val][column] = method

                # Imprimir el resumen detallado
                for isla, mun_data in affected_sections_full_detail.items():
                    print(f"\nIsla: {isla} (Total Municipios Afectados: {len(mun_data)})")
                    for nmun, cusec_data in mun_data.items():
                        print(f"  Municipio: {nmun} (Total Secciones Afectadas: {len(cusec_data)})")
                        for cusec_val, col_imputations in cusec_data.items():
                            imputed_cols_str = ', '.join([f"{col} ({method})" for col, method in col_imputations.items()])
                            print(f"    - CUSEC {cusec_val}: Columnas imputadas: {imputed_cols_str}")

            else:
                print("No se realizó ninguna imputación de valores nulos en las columnas numéricas candidatas.")

            print("\n--- Verificación de Nulos Finales ---")
            final_nan_counts_imputable = merged_df[imputable_cols].isnull().sum()
            final_nan_counts_imputable = final_nan_counts_imputable[final_nan_counts_imputable > 0]
            if not final_nan_counts_imputable.empty:
                print("Advertencia: Aún quedan valores nulos en las siguientes columnas después de la imputación (esto puede deberse a que las columnas 'isla' o 'nmun' estaban nulas para esas filas, o que la imputación global también resultó en NaN si la columna estaba completamente vacía):\n")
                print(final_nan_counts_imputable)
            else:
                print("Todas las columnas numéricas candidatas fueron imputadas exitosamente. No quedan NaNs en ellas.")

### Manejo de anomalías en `distminhospitalkm`

Las filas con `distminhospitalkm <= 0` se consideran anomalías porque una distancia mínima a un hospital debe ser positiva.

La corrección se realiza de forma jerárquica:

1. Mediana de los valores positivos del municipio.
2. Mediana de los valores positivos de la isla si el municipio no permite obtener una mediana válida.
3. Mediana global de los valores positivos como último recurso.

El valor original se conserva en `distminhospitalkm_original` y las filas modificadas quedan identificadas mediante `distminhospitalkm_imputed`.


### Validación de Densidad

Ahora calcularemos `densidad_real_hab_km2 = poblaciontotal / shapearea` y la compararemos con la columna `densidadhabkm2` existente en `merged_df`.

In [ ]:
print("--- Realizando validación y ajuste de densidad ---")

# Desfragmentar el DataFrame para evitar PerformanceWarning
merged_df = merged_df.copy()

# Asegurarse de que las columnas necesarias existen y no tienen nulos críticos para el cálculo
# Ya hemos imputado nulos en 'poblaciontotal' y 'shapearea' si eran numéricas candidatas.
# Aún así, un 'shapearea' de 0 o NaN podría causar problemas. Lo verificaremos.

# Reemplazar 0s en 'shapearea' con NaN para que no causen `inf` y sean manejados si es necesario
# Se usa merged_df['shapearea'] porque ya está limpio de NaNs por la imputación jerárquica.
# Convertir shapearea de m^2 a km^2, asumiendo que la unidad original era m^2 y la densidad es hab/km^2.
merged_df['shapearea_clean'] = merged_df['shapearea'].replace(0, np.nan) / 1_000_000

# **Contexto Importante:** La columna 'shapearea_clean' se ha calculado utilizando la proyección EPSG:25828 (UTM Zona 28N),
# que es la proyección geodésica correcta para las Islas Canarias. Por lo tanto, cualquier densidad calculada con ella
# (`densidad_calculada`) se considera la 'fuente de verdad' si difiere significativamente de la original.

# Calcular densidad real
merged_df['densidad_calculada'] = merged_df['poblaciontotal'] / merged_df['shapearea_clean']

# Comparar con densidadhabkm2
print("Comparando 'densidad_calculada' con 'densidadhabkm2'...")

# Mostrar filas donde hay una diferencia significativa (ej. > 5% o una diferencia absoluta relevante)
difference = (merged_df['densidad_calculada'] - merged_df['densidadhabkm2']).abs()
percentage_difference = (difference / merged_df['densidadhabkm2']).abs() * 100

discrepancy_threshold = 5 # Porcentaje de diferencia para considerar una discrepancia

discrepant_mask = percentage_difference > discrepancy_threshold
discrepant_densities = merged_df[discrepant_mask].copy()

if not discrepant_densities.empty:
    print(f"\nAdvertencia: Se encontraron {len(discrepant_densities)} filas con una diferencia > {discrepancy_threshold}% entre 'densidad_calculada' y 'densidadhabkm2'.")
    display(discrepant_densities.reset_index()[['cusec', 'poblaciontotal', 'shapearea_clean', 'densidad_calculada', 'densidadhabkm2', 'isla', 'nmun']].head())

    print(f"\n--- Reemplazando 'densidadhabkm2' con 'densidad_calculada' para las filas con discrepancia > {discrepancy_threshold}% ---")
    merged_df.loc[discrepant_mask, 'densidadhabkm2'] = merged_df.loc[discrepant_mask, 'densidad_calculada']
    print("Reemplazo completado. La columna 'densidadhabkm2' ahora refleja la densidad calculada para estas secciones.")

    # Verificar si la discrepancia es sistemática en una isla concreta
    print("\n--- Análisis de discrepancias por Isla ---")
    if 'isla' in merged_df.columns:
        discrepancy_by_isla = discrepant_densities.groupby('isla').size().sort_values(ascending=False)
        if not discrepancy_by_isla.empty:
            print("Conteo de CUSECs con discrepancia significativa por isla:\n")
            display(discrepancy_by_isla)
            print("\nSi la discrepancia es sistemática en una isla concreta, podría indicar que el área de esa isla venía en una proyección diferente en el CSV original.")
        else:
            print("No se encontraron discrepancias por isla en los datos discrepantes.")
    else:
        print("Columna 'isla' no encontrada para el análisis por isla.")

else:
    print(f"No se encontraron discrepancias significativas (>{discrepancy_threshold}%) entre 'densidad_calculada' y 'densidadhabkm2'.")

# Resumen estadístico de las diferencias (después del ajuste, si se aplicó)
print("\nResumen estadístico de la diferencia absoluta entre densidades (después del posible ajuste):")
display(difference.describe())

print("\nResumen estadístico de la diferencia porcentual entre densidades (después del posible ajuste):")
display(percentage_difference.describe())

print("--- Fin de la validación y ajuste de densidad ---")

### Validaciones Adicionales Post-Imputación y Consolidación

Continuaremos con validaciones de consistencia para asegurar que las variables porcentuales (`pctextranjeros`, `pctmenor18`, `pctmayor65`) se encuentran dentro de los rangos esperados (0-100) y que sus combinaciones lógicas son válidas, incluso después de la imputación jerárquica y el merge. Esto es crucial para la integridad de los datos antes de la generación de KPIs.

In [ ]:
print("--- Realizando validaciones adicionales post-imputación ---")

# 1. pctextranjeros debe estar en [0, 100] en merged_df
if 'pctextranjeros' in merged_df.columns:
    invalid_pct_extranjeros_post = merged_df[(merged_df['pctextranjeros'] < 0) | (merged_df['pctextranjeros'] > 100)]
    if not invalid_pct_extranjeros_post.empty:
        print(f"\nAdvertencia: Se encontraron {len(invalid_pct_extranjeros_post)} filas con 'pctextranjeros' fuera del rango [0, 100] en merged_df.")
        display(invalid_pct_extranjeros_post[['cusec', 'pctextranjeros', 'isla', 'nmun']].head())
        # Considerar cómo manejar estos valores si persisten. Por ahora, solo advertimos.
    else:
        print("\n'pctextranjeros' en merged_df está dentro del rango [0, 100].")

# 2. pctmenor18 + pctmayor65 no puede superar 100 en merged_df
# Asegurarse de que ambas columnas existen antes de intentar el cálculo
if 'pctmenor18' in merged_df.columns and 'pctmayor65' in merged_df.columns:
    # Rellenar NaNs con 0 temporalmente para el cálculo si hay alguno, para evitar que el resultado sea NaN
    temp_pct_menor18 = merged_df['pctmenor18'].fillna(0)
    temp_pct_mayor65 = merged_df['pctmayor65'].fillna(0)

    invalid_sum_pct_post = merged_df[(temp_pct_menor18 + temp_pct_mayor65) > 100]
    if not invalid_sum_pct_post.empty:
        print(f"\nAdvertencia: Se encontraron {len(invalid_sum_pct_post)} filas donde 'pctmenor18' + 'pctmayor65' > 100 en merged_df.")
        display(invalid_sum_pct_post[['cusec', 'pctmenor18', 'pctmayor65', 'isla', 'nmun']].head())
        # Considerar cómo manejar estos valores si persisten.
    else:
        print("\n'pctmenor18' + 'pctmayor65' en merged_df no supera 100.")
else:
    print("\nAdvertencia: Una o ambas columnas ('pctmenor18', 'pctmayor65') no se encontraron en merged_df para la validación.")

print("\n--- Fin de las validaciones adicionales ---")

### Análisis de Sesgo (Skewness) en Variables Analíticas para Transformación `log1p`

La transformación `log1p` se evaluará únicamente sobre variables numéricas analíticas que puedan formar parte del feature-set. Los identificadores, códigos administrativos, coordenadas y atributos geométricos no se consideran variables cuantitativas del fenómeno y, por tanto, se excluyen del análisis de skewness y de la transformación.

Se utiliza un umbral fijo de skewness de 1.0 para identificar distribuciones con asimetría positiva que puedan beneficiarse de la transformación.

In [ ]:
import numpy as np
from scipy.stats import skew

print('--- Análisis de Sesgo (Skewness) en Variables Analíticas ---')

# Estas columnas son identificadores, códigos administrativos, coordenadas o
# atributos geométricos. No representan variables cuantitativas del fenómeno
# y no deben entrar en el análisis de skewness ni recibir log1p.
NON_ANALYTICAL_NUMERIC_COLS = [
    'cusec', 'cmun', 'csec', 'cdis', 'idregion', 'codigo',
    'objectid', 'objectid1', 'cnut0', 'cnut1', 'cnut2', 'cnut3',
    'anyo', 'cumun', 'cpro', 'cudis', 'clau2',
    'lon', 'lat', 'shapearea', 'shapearea_clean',
    'shapelength', 'shapeleng'
]

numerical_cols_initial = merged_df.select_dtypes(include=np.number).columns.tolist()
analytical_numeric_cols = [
    col for col in numerical_cols_initial
    if col not in NON_ANALYTICAL_NUMERIC_COLS
]

print(f'Columnas numéricas totales: {len(numerical_cols_initial)}')
print(f'Columnas numéricas analíticas evaluadas: {len(analytical_numeric_cols)}')
print(f'Columnas excluidas por no ser variables analíticas: {len(set(numerical_cols_initial) & set(NON_ANALYTICAL_NUMERIC_COLS))}')

skew_values_initial = merged_df[analytical_numeric_cols].apply(
    lambda x: skew(x.dropna()) if len(x.dropna()) > 1 and np.var(x.dropna()) != 0 else np.nan
)
skew_values_initial = skew_values_initial.dropna()

# UMBRAL FIJO: 1.0 en toda la pipeline.
SKEWNESS_THRESHOLD = 1.0

highly_skewed_cols = skew_values_initial[
    skew_values_initial > SKEWNESS_THRESHOLD
].sort_values(ascending=False)
highly_skewed_cols_to_transform = highly_skewed_cols[
    ~highly_skewed_cols.index.str.endswith('_log1p')
]

if not highly_skewed_cols_to_transform.empty:
    print(f'Columnas analíticas con sesgo > {SKEWNESS_THRESHOLD}: {len(highly_skewed_cols_to_transform)}')
    display(highly_skewed_cols_to_transform)
    print(f'\nAplicando np.log1p únicamente a las variables analíticas no negativas con sesgo > {SKEWNESS_THRESHOLD}...')

    for col in highly_skewed_cols_to_transform.index:
        if (merged_df[col].dropna() >= 0).all():
            merged_df[col + '_log1p'] = np.log1p(merged_df[col])
            print(
                f"  '{col}' → '{col}_log1p' | "
                f"sesgo original: {skew_values_initial[col]:.2f} → "
                f"nuevo: {skew(merged_df[col + '_log1p'].dropna()):.2f}"
            )
        else:
            print(f"  SKIP '{col}': contiene valores negativos.")
else:
    print(f'No se encontraron variables analíticas con sesgo > {SKEWNESS_THRESHOLD}.')


### Generación de KPIs Derivados

Vamos a crear una serie de KPIs (Key Performance Indicators) derivados que son fundamentales para el análisis geoespacial y el modelado. Estos KPIs combinan o transforman variables existentes para capturar relaciones más complejas y ofrecer una perspectiva más rica del territorio.

In [ ]:
import numpy as np

print('--- Calculando KPIs derivados ---')
merged_df = merged_df.copy()

# Dependencias
if 'shapearea_clean' not in merged_df.columns and 'shapearea' in merged_df.columns:
    merged_df['shapearea_clean'] = merged_df['shapearea'].replace(0, np.nan)
if 'densidad_calculada' not in merged_df.columns and 'poblaciontotal' in merged_df.columns and 'shapearea_clean' in merged_df.columns:
    merged_df['densidad_calculada'] = merged_df['poblaciontotal'] / merged_df['shapearea_clean']

# KPI 1: Personas por hogar estimadas (ratio adimensional, proxy INE)
merged_df['estimated_persons_per_household'] = (
    merged_df['rentanetamediahogar'] / merged_df['rentanetamediapersona']
)
print("KPI 1 'estimated_persons_per_household' ✓")

# KPI 2: Densidad relativa a la isla (ratio adimensional)
merged_df['relative_island_density'] = (
    merged_df['densidad_calculada'] /
    merged_df.groupby('isla')['densidad_calculada'].transform('mean')
)
print("KPI 2 'relative_island_density' ✓")

# KPI 3: Riqueza relativa del hogar en la isla (ratio adimensional, OCDE privación relativa)
merged_df['relative_island_household_wealth'] = (
    merged_df['rentanetamediahogar'] /
    merged_df.groupby('isla')['rentanetamediahogar'].transform('mean')
)
print("KPI 3 'relative_island_household_wealth' ✓")

# KPI 4: Ratio oferta/demanda sanitaria (OMS, 2010)
merged_df['healthcare_offer_demand_ratio'] = (
    (merged_df['nhospitales5km'] + 1) /
    (merged_df['poblaciontotal'] * (merged_df['pctmenor18'] + merged_df['pctmayor65']) + np.finfo(float).eps)
)
print("KPI 4 'healthcare_offer_demand_ratio' ✓")

# EXCLUIDOS: isolation_index_normalized (suma de km sin normalizar por área)
#            wealth_access_mismatch_score (euros × km, unidades incompatibles)
print("\n[INFO] KPIs isolation_index_normalized y wealth_access_mismatch_score "
      "EXCLUIDOS — combinan variables con unidades incompatibles.")

print('\n--- Resumen KPIs incluidos ---')
display(merged_df[[
    'estimated_persons_per_household',
    'relative_island_density',
    'relative_island_household_wealth',
    'healthcare_offer_demand_ratio'
]].describe())

### Guardando el DataFrame actualizado con los KPIs

Guardaremos el `merged_df` con las nuevas columnas de KPIs en un nuevo archivo CSV para futuras etapas del análisis. Este archivo incluirá todos los datos preprocesados y enriquecidos.

In [ ]:
print("--- Identificando y eliminando columnas con valores constantes ---")

# Identificar columnas con valores constantes (un solo valor único)
constant_columns = [col for col in merged_df.columns if merged_df[col].nunique(dropna=False) == 1]

if constant_columns:
    print(f"Se encontraron {len(constant_columns)} columnas con valores constantes: {constant_columns}")
    # Eliminar estas columnas de merged_df
    merged_df.drop(columns=constant_columns, inplace=True)
    print("Columnas constantes eliminadas de merged_df.")
    print(f"Nuevo shape de merged_df: {merged_df.shape}")
    display(merged_df.head())
else:
    print("No se encontraron columnas con valores constantes en merged_df.")

print("--- Proceso completado ---")

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Local execution: Google Drive mount skipped.')

In [ ]:
print("--- Resumen de los KPIs derivados en merged_df (primeras 5 filas) ---")
display(merged_df[['cusec', 'estimated_persons_per_household',
                   'healthcare_offer_demand_ratio',
                   'tasadependencia']].head())

### Visualización del Sesgo y Efecto de la Transformación `log1p`

Para entender mejor el sesgo de las variables numéricas y el impacto de la transformación `log1p`, visualizaremos las distribuciones de algunas de las columnas más sesgadas antes y después de la transformación. Seleccionaremos las 5 columnas con el sesgo más alto que fueron transformadas.

### Análisis de Correlación entre Variables Numéricas

Vamos a calcular la matriz de correlación para todas las columnas numéricas en `merged_df`. Esto nos permitirá:

1.  **Detectar variables casi idénticas** que añaden ruido sin información nueva (por ejemplo, con un coeficiente de correlación absoluto mayor a 0.95).
2.  **Validar que las relaciones tienen sentido geográfico** (por ejemplo, la renta y la distancia a un hospital deberían correlacionar negativamente).
3.  **Identificar qué variables podrían dominar el espacio latente** de un autoencoder si no se normalizan bien, debido a su alta interdependencia.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

print("--- Calculando matriz de correlación para columnas numéricas ---")

# Seleccionar solo columnas numéricas
numerical_cols_merged = merged_df.select_dtypes(include=np.number)

# Excluir las columnas _log1p para evitar correlaciones directas con las originales
numerical_cols_merged_filtered = numerical_cols_merged.loc[:, ~numerical_cols_merged.columns.str.endswith('_log1p')]

# Calcular la matriz de correlación usando el método de Spearman
correlation_matrix = numerical_cols_merged_filtered.corr(method='spearman')

# --- Identificar pares de columnas altamente correlacionadas (redundancia) ---
correlation_threshold = 0.95

highly_correlated_pairs = []
# Recorrer la mitad superior de la matriz para evitar duplicados y auto-correlación
for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):
        col1 = correlation_matrix.columns[i]
        col2 = correlation_matrix.columns[j]
        corr_value = correlation_matrix.iloc[i, j]

        if abs(corr_value) > correlation_threshold:
            highly_correlated_pairs.append((col1, col2, corr_value))

if highly_correlated_pairs:
    print(f"\nSe encontraron {len(highly_correlated_pairs)} pares de columnas con correlación > {correlation_threshold}:")
    for col1, col2, corr_value in highly_correlated_pairs:
        print(f"  - '{col1}' y '{col2}': {corr_value:.4f}")
else:
    print(f"\nNo se encontraron pares de columnas con correlación > {correlation_threshold}.")


# --- Validar correlaciones esperadas (sentido geográfico/socioeconómico) ---
print("\n--- Correlaciones específicas esperadas ---")

specific_correlations = [
    ('rentanetamediapersona', 'distminhospitalkm'),
    ('rentanetamediahogar', 'distminhospitalkm'),
    ('poblaciontotal', 'shapearea_clean'), # Debería ser alta y positiva
    ('poblaciontotal', 'densidadhabkm2')
]

for col1, col2 in specific_correlations:
    if col1 in correlation_matrix.columns and col2 in correlation_matrix.columns:
        corr_value = correlation_matrix.loc[col1, col2]
        print(f"  - Correlación entre '{col1}' y '{col2}': {corr_value:.4f}")
    else:
        print(f"  - Advertencia: Una o ambas columnas ('{col1}', '{col2}') no se encontraron en la matriz de correlación.")


print("\n--- Mapa de calor de la matriz de correlación (primeras 20 columnas) ---")
# Para una visualización más general, mostrar un mapa de calor para las primeras N columnas
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix.iloc[:20, :20], annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Mapa de Calor de la Matriz de Correlación (Primeras 20 Columnas Numéricas)')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Visualizando distribuciones de columnas sesgadas antes y después de `log1p` ---")

# Obtener las 5 columnas más sesgadas que fueron transformadas
transformed_cols_to_plot = highly_skewed_cols.head(5).index.tolist()

plt.figure(figsize=(15, 10))
for i, col in enumerate(transformed_cols_to_plot):
    # Subplot para la columna original
    plt.subplot(5, 2, 2*i + 1)
    sns.histplot(merged_df[col].dropna(), kde=True)
    plt.title(f'Original: {col} (Skew: {skew_values_initial[col]:.2f})')
    plt.xlabel('')
    plt.ylabel('Frecuencia')

    # Subplot para la columna transformada
    if (col + '_log1p') in merged_df.columns:
        plt.subplot(5, 2, 2*i + 2)
        sns.histplot(merged_df[col + '_log1p'].dropna(), kde=True)
        new_skew = skew(merged_df[col + '_log1p'].dropna())
        plt.title(f'Log1p Transformed: {col}_log1p (Skew: {new_skew:.2f})')
        plt.xlabel('')
        plt.ylabel('Frecuencia')

plt.tight_layout()
plt.show()

print("Visualizaciones completadas.")

In [ ]:
import pandas as pd
import os # Ensure os is imported

print("--- Handling 'object' type columns ---")

# --- Step 0: Robustly reload merged_df to ensure 'isla' and 'nmun' are present and object type ---
# Assuming 'merged_df' from the last stable state (after initial merge) is saved.
# The path used in 'cd54cff2' was 'data/merged_ground_truth_data.csv'
temp_merged_df_path = '../data/outputs/dataset_final.csv'

# Check if merged_df exists in current environment first, if it does, compare its index with 'cusec'
# If it's not indexed by 'cusec' or not present, then reload.
if 'merged_df' in locals() and isinstance(merged_df, pd.DataFrame) and len(merged_df.columns) > 10:
    print('Using existing merged_df from memory.')
elif False:
    if os.path.exists(temp_merged_df_path):
        print(f"Reloading 'merged_df' from {temp_merged_df_path} to ensure 'isla' and 'nmun' integrity.")
        # Temporarily load to check if 'cusec' is index or column
        check_df = pd.read_csv(temp_merged_df_path)
        if 'cusec' in check_df.columns:
            merged_df = check_df.set_index('CUSEC')
        else:
            # If 'cusec' is not a column, assume it's already the index or deal with it later
            merged_df = check_df

        # Ensure 'isla' and 'nmun' are object type after reloading
        if 'isla' in merged_df.columns:
            merged_df['isla'] = merged_df['isla'].astype('object')
        if 'nmun' in merged_df.columns:
            merged_df['nmun'] = merged_df['nmun'].astype('object')
    else:
        print(f"Warning: {temp_merged_df_path} not found. Proceeding with existing 'merged_df'. This might lead to missing 'isla'/'nmun'.")
else:
    print("'merged_df' is already loaded with 'cusec' as index.")

# Identify current 'object' type columns
object_cols = merged_df.select_dtypes(include='object').columns.tolist()

print(f"Current 'object' type columns before processing: {object_cols}")

categorical_for_ohe = ['isla', 'nmun']

# Ensure 'isla' and 'nmun' are indeed object types and present for OHE
categorical_for_ohe_existing = [col for col in categorical_for_ohe if col in merged_df.columns and merged_df[col].dtype == 'object']

# Store original 'isla' and 'nmun' for later use if they are going to be One-Hot Encoded
original_isla_series = None
original_nmun_series = None

if 'isla' in merged_df.columns:
    original_isla_series = merged_df['isla'].copy()
if 'nmun' in merged_df.columns:
    original_nmun_series = merged_df['nmun'].copy()

if categorical_for_ohe_existing:
    print(f"Applying One-Hot Encoding to: {categorical_for_ohe_existing}")
    # Perform One-Hot Encoding on a copy to avoid SettingWithCopyWarning
    merged_df_ohe = pd.get_dummies(merged_df, columns=categorical_for_ohe_existing, prefix=categorical_for_ohe_existing)

    # Re-add original 'isla' and 'nmun' columns to merged_df after OHE, if they were present
    # This step is critical for retaining the original categorical columns for grouping
    if original_isla_series is not None:
        merged_df_ohe['isla'] = original_isla_series
    if original_nmun_series is not None:
        merged_df_ohe['nmun'] = original_nmun_series

    merged_df = merged_df_ohe # Update merged_df with the OHE version
else:
    print("No 'isla' or 'nmun' columns found or they are not of 'object' type for One-Hot Encoding (after reload check).")
    # If OHE is not applied, ensure 'isla' and 'nmun' are still present from the reloaded df
    # This part needs to ensure these columns are not accidentally dropped later if they aren't 'object' for OHE

# After OHE (or if OHE was skipped), re-identify all remaining 'object' columns
remaining_object_cols = merged_df.select_dtypes(include='object').columns.tolist()

# Drop any other remaining object columns that are not 'cusec' (which is the index)
# IMPORTANT: 'isla' and 'nmun' should NOT be dropped here if they were just re-added or were intended to be kept.
# Ensure 'cusec' is handled correctly if it ends up as a column and not index.
cols_to_drop_remaining_objects = [col for col in remaining_object_cols if col != merged_df.index.name and col not in ['isla', 'nmun']]

if cols_to_drop_remaining_objects:
    print(f"Dropping remaining 'object' type columns: {cols_to_drop_remaining_objects}")
    merged_df = merged_df.drop(columns=cols_to_drop_remaining_objects, errors='ignore')
else:
    print("No additional 'object' type columns found to drop.")

print("--- 'Object' column handling completed. New merged_df info: ---")
merged_df.info()

print("\n--- Ensuring geographical coordinates and 'isla' are present ---")
required_geo_cols = ['cusec', 'centroidlon', 'centroidlat', 'isla']
secciones_df_geo = None

if 'normalized_dataframes' in locals() and 'secciones_canarias_isla' in normalized_dataframes:
    secciones_df_geo = normalized_dataframes['secciones_canarias_isla'][['cusec', 'centroidlon', 'centroidlat', 'isla']].copy()
    # Rename for consistency with '_x' suffix logic in map cell
    secciones_df_geo.rename(columns={'centroidlon': 'centroidlon_x', 'centroidlat': 'centroidlat_x'}, inplace=True)
    secciones_df_geo.set_index('cusec', inplace=True)
    print("Geographical data from 'secciones_canarias_isla' prepared.")
else:
    print("Advertencia: 'secciones_canarias_isla' no está disponible en 'normalized_dataframes'. No se pueden asegurar las columnas geográficas.")

# Check if merged_df has the _x suffixed geo columns and 'isla'
missing_geo_cols = [col for col in ['centroidlon_x', 'centroidlat_x', 'isla'] if col not in merged_df.columns]

if missing_geo_cols and secciones_df_geo is not None:
    print(f"Detectadas columnas geográficas faltantes en merged_df: {missing_geo_cols}. Reintegrando desde secciones_canarias_isla.")
    # merged_df is indexed by 'cusec' here
    merged_df = merged_df.merge(secciones_df_geo, left_index=True, right_index=True, how='left', suffixes=('', '_y_dropped'))
    # Drop any unwanted '_y_dropped' columns if they appeared from the merge
    merged_df.drop(columns=[col for col in merged_df.columns if col.endswith('_y_dropped')], inplace=True, errors='ignore')
    print("Columnas geográficas reintegradas. Verificando...")
    # Re-check
    missing_geo_cols_after = [col for col in ['centroidlon_x', 'centroidlat_x', 'isla'] if col not in merged_df.columns]
    if not missing_geo_cols_after:
        print("Todas las columnas geográficas requeridas ahora están presentes.")
    else:
        print(f"Advertencia: Aún faltan columnas geográficas después del intento de reintegración: {missing_geo_cols_after}")
elif not missing_geo_cols:
    print("Todas las columnas geográficas requeridas ya están presentes en merged_df.")
else:
    print("No se pudieron reintegrar las columnas geográficas debido a la falta de datos fuente.")

print("--- Final info after ensuring geo columns ---")
merged_df.info()

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── Feature set explícito y homogéneo para AE, PCA y K-Means ───────────────
# Mismo conjunto de 14 variables temáticas que usa XGBoost.
# Se usan versiones _log1p para variables con sesgo > 1.0 (SKEWNESS_THRESHOLD),
# consistente con el análisis de sesgo de la celda 32.
# Se excluyen: variables económicas, OHE, identificadores, geometría (lat/lon,
# shapearea, objectid, cmun, csec, cdis...) y duplicados de fuentes raw.

FEATURES_TEMATICAS = {
    'salud':      ['distminhospitalkm', 'distmincskm', 'distminfarmaciakm', 'nhospitales5km'],
    'transporte': ['distminparadakm', 'nparadas500m'],
    'demografia': ['poblaciontotal', 'pctextranjeros', 'indiceenvejecimiento',
                   'tasadependencia', 'pctmenor18', 'pctmayor65', 'densidadhabkm2'],
    'educacion':  ['distmincolegiokm'],
}

# Para cada variable: usar versión log1p si existe y tiene sesgo > threshold
FEATURES_NO_ECONOMICAS = []
for dimension, cols in FEATURES_TEMATICAS.items():
    for col in cols:
        log_col = col + '_log1p'
        if log_col in merged_df.columns:
            FEATURES_NO_ECONOMICAS.append(log_col)
            print(f'  [{dimension}] {col} → {log_col} (log1p)')
        elif col in merged_df.columns:
            FEATURES_NO_ECONOMICAS.append(col)
            print(f'  [{dimension}] {col} (original)')
        else:
            print(f'  [{dimension}] WARN: {col} no encontrada en merged_df')

# Añadir los 4 KPIs válidos
KPIS_VALIDOS = [
    'estimated_persons_per_household',
    'relative_island_density',
    'relative_island_household_wealth',
    'healthcare_offer_demand_ratio',
]
for kpi in KPIS_VALIDOS:
    if kpi in merged_df.columns:
        FEATURES_NO_ECONOMICAS.append(kpi)
        print(f'  [KPI] {kpi}')
    else:
        print(f'  [KPI] WARN: {kpi} no encontrado')

print(f'\nTotal features ({len(FEATURES_NO_ECONOMICAS)}): {FEATURES_NO_ECONOMICAS}')

# Preparar X_scaled
X_raw = merged_df[FEATURES_NO_ECONOMICAS].copy()
X_raw = X_raw.fillna(X_raw.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'\nX_scaled shape: {X_scaled.shape}')
print('→ X_scaled listo para Autoencoder, PCA y K-Means.')

In [ ]:
print('--- Verificación de columnas OHE y exclusión de modelos ---\n')

ohe_isla = [col for col in merged_df.columns if col.startswith('isla_')]
ohe_nmun = [col for col in merged_df.columns if col.startswith('nmun_')]
ohe_all  = ohe_isla + ohe_nmun

print(f'Columnas OHE isla:      {len(ohe_isla)}')
print(f'Columnas OHE municipio: {len(ohe_nmun)}')
print(f'Total OHE:              {len(ohe_all)}')
print()
print('[DECISIÓN METODOLÓGICA]')
print('Las columnas OHE de isla y municipio NO se incluyen en el feature set')
print('de ningún modelo (Autoencoder, PCA, K-Means, XGBoost).')
print('Razón: codifican identidad geográfica directa (leakage de identidad).')
print('Permanecen en merged_df solo para análisis descriptivos.')


### Visualización de la Distribución de las Columnas `_log1p` Transformadas

Ahora, vamos a visualizar las distribuciones de algunas de las columnas numéricas a las que se les aplicó la transformación `log1p` para reducir su sesgo. Esto nos ayudará a entender cómo ha cambiado su distribución.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import skew

print("--- Visualizando distribuciones de columnas _log1p transformadas ---")

if 'merged_df' not in locals():
    raise RuntimeError("'merged_df' no encontrado. Por favor, asegúrese de que las celdas de carga y preprocesamiento de datos se hayan ejecutado.")

# Mantener una Serie para que el índice contenga los nombres de las columnas.
if 'highly_skewed_cols' not in locals():
    print("Advertencia: 'highly_skewed_cols' no definida. Recreando para identificar versiones log1p.")
    exclude_cols_from_skew_local = ['cusec', 'objectid', 'objectid1']
    numerical_cols_temp_local = merged_df.select_dtypes(include=np.number).columns.tolist()
    numerical_cols_temp_local = [col for col in numerical_cols_temp_local if col not in exclude_cols_from_skew_local]
    existing_numerical_cols_local = [col for col in numerical_cols_temp_local if col in merged_df.columns]
    if existing_numerical_cols_local:
        skew_values_temp_local = merged_df[existing_numerical_cols_local].apply(
            lambda x: skew(x.dropna()) if len(x.dropna()) > 1 else np.nan
        )
        skewness_threshold_local = 1.0
        highly_skewed_cols = skew_values_temp_local[
            skew_values_temp_local > skewness_threshold_local
        ].sort_values(ascending=False)
    else:
        highly_skewed_cols = pd.Series(dtype='float64')

source_columns = (
    highly_skewed_cols.index.tolist()
    if hasattr(highly_skewed_cols, 'index')
    else list(highly_skewed_cols)
)
transformed_cols_to_plot = [
    f'{col_name}_log1p'
    for col_name in source_columns
    if f'{col_name}_log1p' in merged_df.columns
][:5]

if not transformed_cols_to_plot:
    print("No se encontraron columnas _log1p transformadas para visualizar.")
else:
    plt.figure(figsize=(15, len(transformed_cols_to_plot) * 3))

    for i, col in enumerate(transformed_cols_to_plot):
        plt.subplot(len(transformed_cols_to_plot), 1, i + 1)
        sns.histplot(merged_df[col].dropna(), kde=True)

        original_col_name = col.removesuffix('_log1p')
        original_values = merged_df[original_col_name].dropna()
        transformed_values = merged_df[col].dropna()
        original_skew_val = skew(original_values) if len(original_values) > 1 else np.nan
        new_skew_val = skew(transformed_values) if len(transformed_values) > 1 else np.nan

        plt.title(f'Distribución de {col} (Original Skew: {original_skew_val:.2f}, New Skew: {new_skew_val:.2f})')
        plt.xlabel('')
        plt.ylabel('Frecuencia')

    plt.tight_layout()
    plt.show()

print("Visualización de distribuciones de columnas _log1p transformadas completada.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("--- Calculando matriz de correlación para columnas _log1p transformadas ---")

# Seleccionar solo columnas que terminan con '_log1p'
log1p_cols = [col for col in merged_df.columns if col.endswith('_log1p')]

if not log1p_cols:
    print("No se encontraron columnas transformadas con '_log1p'.")
else:
    print(f"Se encontraron {len(log1p_cols)} columnas transformadas con '_log1p'.")

    # Crear un DataFrame solo con estas columnas
    log1p_df = merged_df[log1p_cols].copy()

    # Calcular la matriz de correlación usando el método de Spearman
    correlation_matrix_log1p = log1p_df.corr(method='spearman')

    print("Matriz de correlación calculada.")
    display(correlation_matrix_log1p.head())

    # --- Identificar pares de columnas altamente correlacionadas (redundancia) ---
    correlation_threshold = 0.95

    highly_correlated_pairs_log1p = []
    # Recorrer la mitad superior de la matriz para evitar duplicados y auto-correlación
    for i in range(len(correlation_matrix_log1p.columns)):
        for j in range(i + 1, len(correlation_matrix_log1p.columns)):
            col1 = correlation_matrix_log1p.columns[i]
            col2 = correlation_matrix_log1p.columns[j]
            corr_value = correlation_matrix_log1p.iloc[i, j]

            if abs(corr_value) > correlation_threshold:
                highly_correlated_pairs_log1p.append((col1, col2, corr_value))

    if highly_correlated_pairs_log1p:
        print(f"\nSe encontraron {len(highly_correlated_pairs_log1p)} pares de columnas _log1p con correlación > {correlation_threshold}:")
        for col1, col2, corr_value in highly_correlated_pairs_log1p:
            print(f"  - '{col1}' y '{col2}': {corr_value:.4f}")
    else:
        print(f"\nNo se encontraron pares de columnas _log1p con correlación > {correlation_threshold}.")

    # --- Validar correlaciones específicas esperadas en datos _log1p ---
    specific_correlations_log1p = [
        ('rentanetamediapersona_log1p', 'distminhospitalkm_log1p'),
        ('rentanetamediahogar_log1p', 'distminhospitalkm_log1p'),
        ('poblaciontotal_log1p', 'shapearea_clean_log1p'),
        ('poblaciontotal_log1p', 'densidadhabkm2_log1p')
    ]

    print("\n--- Correlaciones específicas esperadas en datos _log1p ---")
    for col1, col2 in specific_correlations_log1p:
        if col1 in correlation_matrix_log1p.columns and col2 in correlation_matrix_log1p.columns:
            corr_value = correlation_matrix_log1p.loc[col1, col2]
            print(f"  - Correlación entre '{col1}' y '{col2}': {corr_value:.4f}")
        else:
            print(f"  - Advertencia: Una o ambas columnas ('{col1}', '{col2}') no se encontraron en la matriz de correlación _log1p.")


    print("\n--- Mapa de calor de la matriz de correlación (primeras 20 columnas _log1p) ---")
    plt.figure(figsize=(16, 14))
    sns.heatmap(correlation_matrix_log1p.iloc[:20, :20], annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Mapa de Calor de la Matriz de Correlación (Primeras 20 Columnas _log1p Transformadas)')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

print("Análisis de correlación de columnas _log1p completado.")

In [ ]:
import folium
import pandas as pd
import numpy as np

print("--- Generando mapas para visualizar CUSECs antes y después del merge ---")

# Definir una paleta de colores para las islas
isla_colors = {
    'Gran Canaria': 'blue',
    'Tenerife': 'red',
    'La Palma': 'green',
    'Fuerteventura': 'orange',
    'Lanzarote': 'purple',
    'La Gomera': 'darkgreen',
    'El Hierro': 'cadetblue',
    np.nan: 'gray' # Color para valores nulos de isla
}

# --- Mapa para datos ANTES del merge (dataset_canarias_raw) ---
# Using 'secciones_canarias_isla' from normalized_dataframes as 'dataset_canarias_raw' is not available
if 'normalized_dataframes' in locals() and 'secciones_canarias_isla' in normalized_dataframes:
    df_pre_merge = normalized_dataframes['secciones_canarias_isla'].copy()

    # Asegurar que las columnas existen y son numéricas para las coordenadas
    required_cols_pre = ['cusec', 'centroidlon', 'centroidlat', 'isla']
    if all(col in df_pre_merge.columns for col in required_cols_pre):
        df_pre_merge.dropna(subset=['centroidlon', 'centroidlat'], inplace=True)

        # Centro aproximado de las Islas Canarias para el mapa
        center_lat_pre, center_lon_pre = 28.291565, -16.629130 # Tenerife, al centro
        map_pre_merge = folium.Map(location=[center_lat_pre, center_lon_pre], zoom_start=8)

        for idx, row in df_pre_merge.iterrows():
            isla_color = isla_colors.get(row['isla'], 'gray')
            folium.CircleMarker(
                location=[row['centroidlat'], row['centroidlon']],
                radius=3,
                color=isla_color,
                fill=True,
                fill_color=isla_color,
                fill_opacity=0.7,
                tooltip=f"CUSEC: {row['cusec']}<br>Isla: {row['isla']}"
            ).add_to(map_pre_merge)

        print("\nMapa de CUSECs antes del merge (secciones_canarias_isla):")
        display(map_pre_merge)
    else:
        print("Advertencia: El DataFrame 'secciones_canarias_isla' no contiene todas las columnas necesarias para la visualización (cusec, centroidlon, centroidlat, isla).")
else:
    print("Advertencia: 'normalized_dataframes' o 'secciones_canarias_isla' no están disponibles.")

In [ ]:
# --- Mapa para datos DESPUÉS del merge (merged_df) ---
if 'merged_df' in locals():
    df_post_merge = merged_df.copy()

    # Asegurar que las columnas existen y son numéricas para las coordenadas
    # Usar las columnas con sufijo '_x' que resultaron del merge
    required_cols_post = ['cusec', 'centroidlon_x', 'centroidlat_x', 'isla']
    if all(col in df_post_merge.columns for col in required_cols_post):
        df_post_merge.dropna(subset=['centroidlon_x', 'centroidlat_x'], inplace=True)

        # Centro aproximado de las Islas Canarias para el mapa
        center_lat_post, center_lon_post = 28.291565, -16.629130 # Tenerife, al centro
        map_post_merge = folium.Map(location=[center_lat_post, center_lon_post], zoom_start=8)

        for idx, row in df_post_merge.iterrows():
            isla_color = isla_colors.get(row['isla'], 'gray')
            folium.CircleMarker(
                location=[row['centroidlat_x'], row['centroidlon_x']], # Usar columnas con sufijo '_x'
                radius=3,
                color=isla_color,
                fill=True,
                fill_color=isla_color,
                fill_opacity=0.7,
                tooltip=f"CUSEC: {row['cusec']}<br>Isla: {row['isla']}"
            ).add_to(map_post_merge)

        print("\nMapa de CUSECs después del merge (merged_df):")
        display(map_post_merge)
    else:
        print("Advertencia: El DataFrame 'merged_df' no contiene todas las columnas necesarias para la visualización (cusec, centroidlon_x, centroidlat_x, isla).")
else:
    print("Advertencia: 'merged_df' no está disponible.")

print("\n--- Visualización de mapas completada ---")

### Comparación de Correlaciones: Datos Originales vs. Transformados `_log1p`

Para entender el impacto de la transformación `log1p` en las relaciones entre variables, compararemos las matrices de correlación de las columnas originales que fueron identificadas como altamente sesgadas con sus versiones transformadas `_log1p`. Esto nos permitirá observar cómo la transformación altera las interdependencias lineales entre estas características.

In [ ]:
print("--- Determinando el número de columnas originales y transformadas ---")

# Asegurarse de que merged_df está disponible
if 'merged_df' not in locals():
    raise RuntimeError("'merged_df' no encontrado. Por favor, asegúrese de que las celdas de carga y preprocesamiento de datos se hayan ejecutado.")

# Initialize counters
transformed_log1p_cols_count = 0
original_untransformed_cols_count = 0
original_transformed_base_cols_count = 0

# Keep track of original column names that correspond to a _log1p version
original_cols_with_log1p_version = set()

for col in merged_df.columns:
    if col.endswith('_log1p'):
        transformed_log1p_cols_count += 1
        original_name = col.replace('_log1p', '')
        if original_name in merged_df.columns:
            original_cols_with_log1p_version.add(original_name)

# Now iterate again to count truly untransformed originals
for col in merged_df.columns:
    if not col.endswith('_log1p'): # It's not a _log1p column itself
        if col not in original_cols_with_log1p_version: # And it's not the original for a _log1p column
            # This means it's an original column that was never transformed
            original_untransformed_cols_count += 1

original_transformed_base_cols_count = len(original_cols_with_log1p_version)

# Total original columns in a broad sense (those that are not _log1p versions)
# This is the sum of originals that *were* transformed and originals that *were not* transformed
total_original_cols_in_df = original_transformed_base_cols_count + original_untransformed_cols_count

print(f"Número total de columnas transformadas con '_log1p' (columnas nuevas): {transformed_log1p_cols_count}")
print(f"Número de columnas originales para las que se creó una versión '_log1p': {original_transformed_base_cols_count}")
print(f"Número de columnas originales que NO fueron transformadas con '_log1p': {original_untransformed_cols_count}")
print(f"Número total de columnas base (originales + originales con versión '_log1p'): {total_original_cols_in_df}")
print(f"Verificación: Total de columnas en merged_df: {merged_df.shape[1]}")
print(f"Verificación: Suma de (columnas transformadas) + (columnas base no transformadas): {transformed_log1p_cols_count + original_untransformed_cols_count + original_transformed_base_cols_count}")

print("--- Conteo de columnas completado ---")

In [ ]:
# ── K-Means usa X_scaled definido en celda 43 (feature set homogéneo de 18 variables)
# No se prepara un X_scaled propio: garantiza comparabilidad con AE y PCA.
import numpy as np
from sklearn.preprocessing import StandardScaler

print('--- Preparando datos para Clustering K-Means ---')
print(f'Usando X_scaled compartido: shape={X_scaled.shape}')
print(f'Features ({len(FEATURES_NO_ECONOMICAS)}): {FEATURES_NO_ECONOMICAS}')

# X_scaled_kmeans apunta al mismo array que usan AE y PCA
X_scaled_kmeans = X_scaled.copy()
print('\nDatos listos para K-Means (mismo feature set que AE y PCA).')


### Clustering Exploratorio K-Means

Para preparar los datos para el clustering K-Means, seleccionaremos las columnas numéricas relevantes. Priorizaremos las columnas que ya han sido transformadas con `log1p` para reducir el sesgo, ya que los algoritmos basados en distancia como K-Means funcionan mejor con distribuciones más simétricas. Excluiremos columnas de identificación como `cusec`, `isla`, `nmun`, y `objectid` ya que no son características numéricas para el clustering, sino que se usarán para analizar los resultados.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

print("--- Aplicando Método del Codo ---")

wcss = []
max_k = 10 # Se puede ajustar según el tamaño del dataset y la lógica de negocio

if 'X_scaled' in locals():
    for i in range(1, max_k + 1):
        kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10) # n_init para evitar warnings
        kmeans.fit(X_scaled)
        wcss.append(kmeans.inertia_)

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, max_k + 1), wcss, marker='o', linestyle='--')
    plt.title('Método del Codo para K Óptimo')
    plt.xlabel('Número de Clusters (K)')
    plt.ylabel('WCSS (Suma de Cuadrados Intra-Cluster)')
    plt.xticks(range(1, max_k + 1))
    plt.grid(True)
    plt.show()

    print("Identifica el 'codo' en el gráfico para estimar el número óptimo de clusters.")
else:
    print("Error: 'X_scaled' no está definido. Asegúrate de que la celda de preparación de datos se haya ejecutado correctamente.")

#### Método de la Silueta para determinar K

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import pandas as pd

print('--- Selección de K: Silhouette Score + Davies-Bouldin Index ---')

max_k = 10
silhouette_scores = []
davies_bouldin_scores = []
k_range = range(2, max_k + 1)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    db  = davies_bouldin_score(X_scaled, labels)
    silhouette_scores.append(sil)
    davies_bouldin_scores.append(db)
    print(f'  K={k:2d}  |  Silhouette={sil:.4f}  |  Davies-Bouldin={db:.4f}')

results_df = pd.DataFrame({
    'K': list(k_range),
    'Silhouette': silhouette_scores,
    'Davies-Bouldin': davies_bouldin_scores
})
print('\n--- Tabla resumen (Silhouette: mayor=mejor | Davies-Bouldin: menor=mejor) ---')
display(results_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(list(k_range), silhouette_scores, marker='o', color='steelblue')
ax1.axvline(x=4, color='red', linestyle='--', label='K=4 seleccionado')
ax1.set_title('Silhouette Score (mayor = mejor)')
ax1.set_xlabel('K'); ax1.set_ylabel('Silhouette Score')
ax1.legend(); ax1.grid(True)

ax2.plot(list(k_range), davies_bouldin_scores, marker='o', color='darkorange')
ax2.axvline(x=4, color='red', linestyle='--', label='K=4 seleccionado')
ax2.set_title('Davies-Bouldin Index (menor = mejor)')
ax2.set_xlabel('K'); ax2.set_ylabel('Davies-Bouldin')
ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()

k_sil_opt = results_df.loc[results_df['Silhouette'].idxmax(), 'K']
k_db_opt  = results_df.loc[results_df['Davies-Bouldin'].idxmin(), 'K']
print(f'\n  K óptimo por Silhouette:     K={k_sil_opt}')
print(f'  K óptimo por Davies-Bouldin: K={k_db_opt}')

print('''
[DECISIÓN METODOLÓGICA — K=4]
Ningún índice cuantitativo señala K=4 como óptimo:
  - Silhouette máximo: K=2
  - Davies-Bouldin mínimo: K=2

La elección de K=4 se justifica exclusivamente por criterios territoriales:
  1. K=2 y K=3 colapsan perfiles territorialmente distintos en un único cluster
     (islas menores + periferias vulnerables de islas capitalinas).
  2. K=5 produce clusters con < 10 secciones en islas menores (El Hierro,
     La Gomera), insuficientes para interpretación estadística fiable.
  3. K=4 reproduce los cuatro perfiles identificados en la literatura de
     tipologías socioterritoriales insulares (OCDE, 2008; Eurostat, 2018):
     núcleo urbano, periurbano conectado, rural envejecido y periferia aislada.

Esta justificación territorial debe documentarse explícitamente en la memoria.
''')

print('\n⚠ ANOTAR PARA EL DOCUMENTO:')
for _, row in results_df[results_df['K'].isin([2, 3, 4, 5])].iterrows():
    print(f'  K={int(row["K"])}:  Silhouette={row["Silhouette"]:.4f}  |  DB={row["Davies-Bouldin"]:.4f}')

optimal_k = 4
print(f'\noptimal_k = {optimal_k} (fijado por criterio territorial)')

#### Aplicar K-Means con K Óptimo y Análisis de Clústeres

In [ ]:
optimal_k = 4 # @param {type:"integer"}

print(f"--- Aplicando K-Means con K = {optimal_k} ---")

if 'X_scaled' in locals():
    kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
    merged_df['cluster_label'] = kmeans.fit_predict(X_scaled)

    print(f"K-Means completado con {optimal_k} clusters.")

    print("\n--- Distribución de Clústeres por Isla ---")
    # Asegurarse de que 'isla' no tiene nulos para este análisis o imputarlos si es necesario
    cluster_distribution = merged_df.groupby(['isla', 'cluster_label']).size().unstack(fill_value=0)
    display(cluster_distribution)

    print("\n--- Visualización de la Distribución de Clústeres por Isla ---")
    cluster_distribution.plot(kind='bar', stacked=True, figsize=(12, 7))
    plt.title(f'Distribución de Secciones Censales en Clusters por Isla (K={optimal_k})')
    plt.xlabel('Isla')
    plt.ylabel('Número de Secciones Censales')
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Cluster')
    plt.tight_layout()
    plt.show()

    print("\n--- Estadísticas Descriptivas de los Clústeres ---")
    # Para entender mejor cada cluster, podemos ver las estadísticas de las características originales
    # o las transformadas para cada cluster. Usaremos un subconjunto relevante de features.
    # Se han eliminado las características que no estaban presentes en el `merged_df` debido a la recarga previa del DataFrame
    # en la celda `5sxJPckFb_YA`, ya que no fueron guardadas en el CSV y se perdieron.
    relevant_features_for_description = ['poblaciontotal', 'rentanetamediahogar',
                                         'tasadependencia']

    cluster_summary = merged_df.groupby('cluster_label')[relevant_features_for_description].mean()
    display(cluster_summary)

    print("El análisis de estas estadísticas y la distribución por isla permite interpretar las tipologías territoriales emergentes.")
else:
    print("Error: 'X_scaled' no está definido. Asegúrate de que la celda de preparación de datos se haya ejecutado correctamente.")

### Aplicación de K-Means sobre el Espacio Latente del Autoencoder

Ahora aplicaremos K-Means sobre el espacio latente (`latent_space_df`) extraído del Autoencoder para ver cómo se agrupan las tipologías territoriales en esta representación no lineal.

In [ ]:
print(f"--- Aplicando K-Means con K = {optimal_k} al Espacio Latente del Autoencoder ---")

if 'latent_space_df' in locals() and not latent_space_df.empty:
    kmeans_ae = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
    merged_df['cluster_label_ae'] = kmeans_ae.fit_predict(latent_space_df)

    print(f"K-Means completado sobre el espacio latente del Autoencoder con {optimal_k} clusters.")

    # Calcular Silhouette Score para K-Means en el espacio latente del AE
    silhouette_ae_latent = silhouette_score(latent_space_df, merged_df['cluster_label_ae'])
    print(f"Silhouette Score para K-Means en el espacio latente del Autoencoder: {silhouette_ae_latent:.4f}")

    print("\n--- Distribución de Clústeres del Autoencoder por Isla ---")
    cluster_distribution_ae = merged_df.groupby(['isla', 'cluster_label_ae']).size().unstack(fill_value=0)
    display(cluster_distribution_ae)

    print("\n--- Estadísticas Descriptivas de los Clústeres del Autoencoder ---")
    # Para entender mejor cada cluster, podemos ver las estadísticas de las características originales
    # que componen el espacio latente. Usamos X_scaled_final_df para las características.
    # Filtramos por las características que realmente se usaron para entrenar el AE
    relevant_features_ae_desc = [col for col in X_scaled_final_df.columns if col not in ['cusec', 'isla', 'nmun']]
    cluster_summary_ae = merged_df.groupby('cluster_label_ae')[relevant_features_ae_desc].mean()
    display(cluster_summary_ae.head())

    # Calcular Moran's I para los cluster_label_ae
    if 'w' in locals() and 'calcular_morans_i' in locals():
        cluster_labels_ae_aligned = merged_df['cluster_label_ae'].loc[w.id_order].copy()
        cluster_ae_df_for_moran = pd.DataFrame(cluster_labels_ae_aligned, columns=['cluster_label_ae'])
        moran_kmeans_ae_results = calcular_morans_i(cluster_ae_df_for_moran, w)
        moran_kmeans_ae_df = pd.DataFrame.from_dict(moran_kmeans_ae_results, orient='index')
        moran_kmeans_ae_value = moran_kmeans_ae_df.loc['cluster_label_ae', 'I']
        print(f"Moran's I para Etiquetas de Clúster de Autoencoder: {moran_kmeans_ae_value:.4f}")
    else:
        print("Advertencia: 'w' o 'calcular_morans_i' no están disponibles. No se pudo calcular Moran's I para clusters del Autoencoder.")
        moran_kmeans_ae_value = np.nan

else:
    print("Error: 'latent_space_df' no está definido o está vacío. Asegúrate de que el Autoencoder se haya ejecutado correctamente.")
    silhouette_ae_latent = np.nan
    moran_kmeans_ae_value = np.nan

### Aplicación de K-Means sobre los Componentes PCA

Ahora haremos lo mismo para los componentes principales de PCA (`pca_df`), lo que nos dará una línea base de clustering lineal para comparar.

In [ ]:
print(f"--- Aplicando K-Means con K = {optimal_k} a los Componentes PCA ---")

if 'pca_df' in locals() and not pca_df.empty:
    kmeans_pca = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
    merged_df['cluster_label_pca'] = kmeans_pca.fit_predict(pca_df)

    print(f"K-Means completado sobre los componentes PCA con {optimal_k} clusters.")

    # Calcular Silhouette Score para K-Means en el espacio PCA
    silhouette_pca_components = silhouette_score(pca_df, merged_df['cluster_label_pca'])
    print(f"Silhouette Score para K-Means en los Componentes PCA: {silhouette_pca_components:.4f}")

    print("\n--- Distribución de Clústeres de PCA por Isla ---")
    cluster_distribution_pca = merged_df.groupby(['isla', 'cluster_label_pca']).size().unstack(fill_value=0)
    display(cluster_distribution_pca)

    print("\n--- Estadísticas Descriptivas de los Clústeres de PCA ---")
    # Usamos X_scaled_final_df para las características para el resumen descriptivo
    relevant_features_pca_desc = [col for col in X_scaled_final_df.columns if col not in ['cusec', 'isla', 'nmun']]
    cluster_summary_pca = merged_df.groupby('cluster_label_pca')[relevant_features_pca_desc].mean()
    display(cluster_summary_pca.head())

    # Calcular Moran's I para los cluster_label_pca
    if 'w' in locals() and 'calcular_morans_i' in locals():
        cluster_labels_pca_aligned = merged_df['cluster_label_pca'].loc[w.id_order].copy()
        cluster_pca_df_for_moran = pd.DataFrame(cluster_labels_pca_aligned, columns=['cluster_label_pca'])
        moran_kmeans_pca_results = calcular_morans_i(cluster_pca_df_for_moran, w)
        moran_kmeans_pca_df = pd.DataFrame.from_dict(moran_kmeans_pca_results, orient='index')
        moran_kmeans_pca_value = moran_kmeans_pca_df.loc['cluster_label_pca', 'I']
        print(f"Moran's I para Etiquetas de Clúster de PCA: {moran_kmeans_pca_value:.4f}")
    else:
        print("Advertencia: 'w' o 'calcular_morans_i' no están disponibles. No se pudo calcular Moran's I para clusters de PCA.")
        moran_kmeans_pca_value = np.nan

else:
    print("Error: 'pca_df' no está definido o está vacío. Asegúrate de que PCA se haya ejecutado correctamente.")
    silhouette_pca_components = np.nan
    moran_kmeans_pca_value = np.nan

### Actualización de la Tabla de Benchmark Final con Resultados de K-Means Detallados

Ahora que hemos calculado los resultados de K-Means para los tres inputs diferentes, actualizaremos la tabla de benchmark para reflejar estas comparaciones, incluyendo los Silhouette Scores y los valores de Moran's I para los clusters de cada enfoque.

In [ ]:
print("--- Compilando la Tabla de Benchmark Final --- ")

# Autoencoder: reconstruction loss y promedio Moran's I (promedio por isla)
# ae_reconstruction_loss_spatial_kf from previous spatial CV cell
if 'ae_reconstruction_loss_spatial_kf' in locals():
    ae_reconstruction_loss = ae_reconstruction_loss_spatial_kf
else:
    print("Advertencia: 'ae_reconstruction_loss_spatial_kf' no definida. Usando el 'val_loss' final del history si existe.")
    ae_reconstruction_loss = history.history['val_loss'][-1] if 'history' in locals() and hasattr(history, 'history') and 'val_loss' in history.history else "N/A"

# Calcular el promedio de Moran's I para el Autoencoder de los resultados ya calculados (per-island average)
if 'moran_ae_df' in locals() and not moran_ae_df.empty:
    average_moran_ae_per_island = moran_ae_df['Moran_I'].mean()
else:
    average_moran_ae_per_island = None
    print("Advertencia: 'moran_ae_df' no encontrado o vacío. No se pudo calcular el promedio Moran's I para AE.")

# PCA: explained variance y promedio Moran's I (promedio por isla)
pca_explained_variance = pca.explained_variance_ratio_.sum() if 'pca' in locals() and hasattr(pca, 'explained_variance_ratio_') else "N/A"

# Calcular el promedio de Moran's I para PCA de los resultados ya calculados (per-island average)
if 'moran_pca_df' in locals() and not moran_pca_df.empty:
    average_moran_pca_per_island = moran_pca_df['Moran_I'].mean()
else:
    average_moran_pca_per_island = None
    print("Advertencia: 'moran_pca_df' no encontrado o vacío. No se pudo calcular el promedio Moran's I para PCA.")

# K-Means (3 entradas diferentes):
# - Original Scaled Data
# - Autoencoder Latent Space
# - PCA Components

# Existing K-Means on X_scaled_final_df (Original Data)
# silhouette_kmeans should correspond to X_scaled_final_df from 'a2e8b531'
if 'silhouette_kmeans_original' in locals():
    silhouette_kmeans_original_data = silhouette_kmeans_original
else:
    silhouette_kmeans_original_data = silhouette_scores[optimal_k-2] if 'silhouette_scores' in locals() and optimal_k >= 2 and optimal_k-2 < len(silhouette_scores) else "N/A"

# Moran's I for K-Means on original data
if 'moran_kmeans_original_value' in locals():
    moran_kmeans_original_data_value = moran_kmeans_original_value
else:
    moran_kmeans_original_data_value = moran_kmeans_df['Moran_I'].mean() if 'moran_kmeans_df' in locals() and not moran_kmeans_df.empty else None

# K-Means on AE Latent Space
silhouette_kmeans_ae_latent = silhouette_ae_latent if 'silhouette_ae_latent' in locals() else "N/A"
moran_kmeans_ae_latent_value = moran_kmeans_ae_value if 'moran_kmeans_ae_value' in locals() else None

# K-Means on PCA Components
silhouette_kmeans_pca_components = silhouette_pca_components if 'silhouette_pca_components' in locals() else "N/A"
moran_kmeans_pca_components_value = moran_kmeans_pca_value if 'moran_kmeans_pca_value' in locals() else None


# XGBoost: R-squared y MSE (usando los valores base corregidos por leakage)
# Estos valores ya fueron actualizados en la celda anterior de ablación económica.
xgboost_r2_final = xgboost_r2_base if 'xgboost_r2_base' in locals() else "N/A"
xgboost_mse_final = xgboost_mse_base if 'xgboost_mse_base' in locals() else "N/A"

# Prepare data for the benchmark table
benchmark_data_final = {
    'Modelo': [
        'Autoencoder (Reconstrucción)',
        'Autoencoder (Latent K-Means)',
        'PCA (Varianza)',
        'PCA (Component K-Means)',
        'K-Means (Original Data)',
        'XGBoost'
    ],
    'Métrica Principal 1': [
        'Reconstruction Loss',
        'Silhouette Score',
        'Explained Variance',
        'Silhouette Score',
        'Silhouette Score',
        'R-squared'
    ],
    'Valor Métrica 1': [
        ae_reconstruction_loss,
        silhouette_kmeans_ae_latent,
        pca_explained_variance,
        silhouette_kmeans_pca_components,
        silhouette_kmeans_original_data,
        xgboost_r2_final
    ],
    'Métrica Principal 2': [
        "Moran's I Promedio (Latent)",
        "Moran's I para Clusters",
        "Moran's I Promedio (Comp.)",
        "Moran's I para Clusters",
        "Moran's I para Clusters",
        'MSE'
    ],
    'Valor Métrica 2': [
        average_moran_ae_per_island,
        moran_kmeans_ae_latent_value,
        average_moran_pca_per_island,
        moran_kmeans_pca_components_value,
        moran_kmeans_original_data_value,
        xgboost_mse_final
    ]
}

benchmark_table_final = pd.DataFrame(benchmark_data_final)
display(benchmark_table_final)

print("--- Tabla de Benchmark Final Generada ---")

### Perfiles de Clústeres (Características Medias)

Para interpretar la naturaleza de cada clúster, analizaremos las estadísticas descriptivas (la media) de las características clave para cada grupo. Esto nos permitirá inferir los perfiles socioeconómicos y geográficos predominantes en cada tipología territorial identificada.

In [ ]:
print("--- Calculando el perfil de las características medias para cada clúster ---")

# Asegurarse de que merged_df está cargado y tiene 'cluster_label'
if 'merged_df' not in locals():
    raise RuntimeError("'merged_df' no está disponible. Asegúrate de ejecutar las celdas de carga de datos.")

if 'cluster_label' not in merged_df.columns:
    raise RuntimeError("'cluster_label' no encontrado en 'merged_df'. Asegúrate de que K-Means se ha ejecutado.")

# Define clustering_features_df using X_raw (which contains the features used for clustering)
# X_raw is created in cell 'V4dwonW3-4HU' from FEATURES_NO_ECONOMICAS.
# Ensure its index aligns with merged_df for proper concatenation later.
if 'X_raw' in locals() and isinstance(X_raw, pd.DataFrame):
    clustering_features_df = X_raw.copy()
    # Ensure the index is aligned for concatenation, assuming X_raw was built from merged_df
    # If merged_df index is cusec, and X_raw has a default integer index, reset X_raw's index to cusec
    if merged_df.index.name == 'cusec' and 'cusec' in merged_df.columns and X_raw.index.name != 'cusec':
        # This assumes X_raw rows correspond to merged_df rows, which they should if created sequentially.
        # Re-indexing X_raw to match merged_df's cusec index for a robust join.
        clustering_features_df.index = merged_df.index # Align indices
else:
    raise RuntimeError("'X_raw' no está disponible o no es un DataFrame. Asegúrate de que la celda de preparación de datos para clustering se ha ejecutado correctamente.")

# Asegurarse de que clustering_features_df está disponible y no está vacío
if clustering_features_df.empty:
    raise RuntimeError("'clustering_features_df' está vacío después de su creación. Revisa la fuente de datos.")

# Combinar las características de clustering con las etiquetas de clúster.
# Se realiza una unión por índice (asumiendo que tanto clustering_features_df como merged_df están indexados por cusec)
# Given that merged_df already has the 'cluster_label' column and clustering_features_df is derived from merged_df,
# we can simply select the relevant columns from merged_df.
cluster_profile_data = merged_df[clustering_features_df.columns.tolist() + ['cluster_label']].copy()

# Calcular la media de cada característica por clúster
cluster_summary = cluster_profile_data.groupby('cluster_label').mean()

print("Perfil de características medias por clúster (basado en características de clustering):")
display(cluster_summary)

print("\n--- Distribución de Clústeres por Isla (Conteo) ---")
# Se utiliza merged_df para la distribución por isla, ya que contiene 'isla'
cluster_distribution_by_isla = merged_df.groupby(['isla', 'cluster_label']).size().unstack(fill_value=0)
display(cluster_distribution_by_isla)

print("\nInterpretación:")
print("Para interpretar cada clúster, compare los valores medios de las características con el promedio general de los datos. Por ejemplo:")
print("- **Población/Densidad:** Clústeres con valores altos podrían representar zonas urbanas/densas.")
print("- **Renta:** Clústeres con alta 'rentanetamediahogar_log1p' o 'rentanetamediapersona_log1p' indican mayor riqueza.")
print("- **Accesibilidad/Distancias:** Valores bajos en 'distminhospitalkm' o altos en 'nparadas500m' sugieren buena accesibilidad.")
print("- **Demografía:** 'pctextranjeros', 'indiceenvejecimiento' y 'tasadependencia' revelan el tipo de población.")
print("Analice también la distribución de los clústeres entre las diferentes islas para identificar patrones geográficos.")

In [ ]:
print("--- Asignando nombres descriptivos a los clústeres ---")

# Definir el mapeo de etiquetas numéricas a nombres descriptivos
cluster_name_mapping = {
    0: 'Zonas Rurales/Baja Densidad',
    1: 'Áreas Urbanas/Ricas',
    2: 'Áreas Vulnerables/Menos Pobladas',
    3: 'Zonas Aisladas/Rurales de Montaña'
}

# Crear una nueva columna con los nombres de los clústeres
merged_df['cluster_name'] = merged_df['cluster_label'].map(cluster_name_mapping)

print("Columna 'cluster_name' añadida a merged_df.")

print("\n--- Distribución de Clústeres por Isla (con nombres) ---")
cluster_distribution_named = merged_df.groupby(['isla', 'cluster_name']).size().unstack(fill_value=0)
display(cluster_distribution_named)

print("\n--- Perfil de características medias por clúster (con nombres) ---")
# Re-calculate cluster_profile_data using the named clusters
# Ensure clustering_features list is available. It is defined in 'd6d7037e'.
if 'clustering_features' not in locals() or not clustering_features:
    print("Advertencia: 'clustering_features' no definida. Recreando a partir de la lógica original.")
    # This part duplicates logic from d6d7037e - base_clustering_features, highly_skewed_cols
    if 'highly_skewed_cols' not in locals():
        print("Warning: 'highly_skewed_cols' not defined. Attempting to recreate.")
        exclude_cols_from_skew_temp = ['cusec', 'objectid', 'objectid1']
        numerical_cols_temp_current = merged_df.select_dtypes(include=np.number).columns.tolist()
        numerical_cols_temp_current = [col for col in numerical_cols_temp_current if col not in exclude_cols_from_skew_temp]
        existing_numerical_cols_current = [col for col in numerical_cols_temp_current if col in merged_df.columns]
        if not existing_numerical_cols_current:
            skew_values_temp_current = pd.Series(dtype='float64')
        else:
            skew_values_temp_current = merged_df[existing_numerical_cols_current].apply(lambda x: skew(x.dropna()))
        skewness_threshold_current = 1.0
        highly_skewed_cols = skew_values_temp_current[skew_values_temp_current > skewness_threshold_current].index.tolist()
        print("'highly_skewed_cols' recreated for consistency.")

    base_clustering_features_temp = [
        'poblaciontotal', 'densidad_calculada', 'rentanetamediahogar', 'rentanetamediapersona',
        'pctextranjeros', 'pctnacidosextranjero', 'indiceenvejecimiento', 'tasadependencia',
        'pensionmedia', 'pctingresosbajos7500', 'pctriesgopobreza60', 'pctingresosaltos200',
        'indicegini', 'spersonas',
        'distminhospitalkm', 'distmincskm', 'distminfarmaciakm', 'distmincolegiokm',
        'distminparadakm', 'nparadas500m', 'nhospitales5km',
        'estimated_persons_per_household',
        'relative_island_density', 'relative_island_household_wealth',
        'healthcare_offer_demand_ratio',
        'demographic_service_pressure'
    ]

    clustering_features = []
    for col in base_clustering_features_temp:
        log1p_col_name = col + '_log1p'
        if log1p_col_name in merged_df.columns and col in highly_skewed_cols:
            clustering_features.append(log1p_col_name)
        elif col in merged_df.columns:
            clustering_features.append(col)
    clustering_features = list(set([f for f in clustering_features if f in merged_df.select_dtypes(include=np.number).columns]))
    print("'clustering_features' list recreated.")

existing_clustering_features = [col for col in clustering_features if col in merged_df.columns]
cluster_profile_data_named = merged_df[existing_clustering_features + ['cluster_name']].copy()
cluster_summary_named = cluster_profile_data_named.groupby('cluster_name')[existing_clustering_features].mean()
display(cluster_summary_named)

### Interpretación del Perfil de Clústeres y Selección de K

Los K-Means clusters identificados (K=4) representan diferentes tipologías territoriales dentro de las Islas Canarias, caracterizadas por sus perfiles socioeconómicos y geográficos. A continuación, se detalla la interpretación de cada clúster basada en las medias de las características.

#### Selección del Número Óptimo de Clústeres (K)

El método del codo y el coeficiente de silueta se utilizaron para determinar el número óptimo de clústeres. Aunque el método del codo no mostró un punto de inflexión muy pronunciado, el coeficiente de silueta, que mide cuán similar es un objeto a su propio clúster en comparación con otros clústeres, indicó que **K=4** fue una opción razonable, mostrando un coeficiente de **0.1872**. Si bien este valor no es extremadamente alto, es el que mejor equilibrio ofrece en la separación de los clústeres en el rango explorado (ver gráfica del coeficiente de silueta en la celda anterior). Un coeficiente de silueta cercano a 0 indica que los clústeres no están claramente separados, lo cual es común en datos complejos como los socioeconómicos, donde las transiciones entre tipologías territoriales pueden ser graduales. Esta elección de K=4 se alinea con la hipótesis de que existen alrededor de cuatro tipologías territoriales principales: zonas rurales, urbanas de alta densidad, áreas de vulnerabilidad socioeconómica y regiones más aisladas.

#### Perfil de Cada Clúster

*   **Clúster "Zonas Rurales/Baja Densidad" (Anteriormente Clúster 0): Zonas Rurales y de Baja Densidad con Buena Accesibilidad (Principalmente Tenerife)**
    *   **Demografía y Densidad:** Baja población y densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros y nacidos en el extranjero, bajo índice de envejecimiento y alta tasa de dependencia. Esto sugiere poblaciones más pequeñas, posiblemente rurales o envejecidas.
    *   **Economía:** Renta neta media por hogar y persona (`rentanetamediahogar_log1p`, `rentanetamediapersona`) en rangos intermedios. Bajo riesgo de pobreza.
    *   **Accesibilidad y Servicios:** Valores bajos en distancias a servicios esenciales (`distminhospitalkm_log1p`, `distmincskm_log1p`, `distminfarmaciakm_log1p`, `distmincolegiokm_log1p`). Esto indica buena accesibilidad a servicios a pesar de la baja densidad, posiblemente por la proximidad de núcleos urbanos o una infraestructura de servicios eficiente.
    *   **Distribución Insular:** Predominante en Tenerife, con presencia significativa en Gran Canaria.
    *   **Tipología:** "Zona rural o semi-urbana, bien conectada, con una población relativamente estable y menos diversidad demográfica."

*   **Clúster "Áreas Urbanas/Ricas" (Anteriormente Clúster 1): Áreas Urbanas y Densas con Mayor Población y Riqueza (Repartido entre islas principales)**
    *   **Demografía y Densidad:** Alta población (`poblaciontotal_log1p`) y la mayor densidad (`densidad_calculada_log1p`). Mayor porcentaje de extranjeros y nacidos en el extranjero. Tasa de dependencia más baja. Esto indica áreas urbanas, densamente pobladas y dinámicas.
    *   **Economía:** La renta neta media por hogar y persona más alta (`rentanetamediahogar_log1p`, `rentanetamediapersona`), bajo riesgo de pobreza y bajo porcentaje de ingresos bajos. Esto sugiere zonas con mayor riqueza y oportunidades económicas.
    *   **Accesibilidad y Servicios:** Distancias a servicios bajas (`distminhospitalkm_log1p`, etc.) y alto número de paradas de transporte público (`nparadas500m_log1p`). Esto confirma su carácter urbano y bien servido.
    *   **Distribución Insular:** Presente en todas las islas principales (Tenerife, Gran Canaria, Lanzarote).
    *   **Tipología:** "Núcleos urbanos principales, centros económicos y residenciales, con una población diversa y altos niveles de vida."

*   **Clúster "Áreas Vulnerables/Menos Pobladas" (Anteriormente Clúster 2): Zonas con Menor Población, Mayor Vulnerabilidad y Accesibilidad Intermedia (Principalmente Gran Canaria y Tenerife)**
    *   **Demografía y Densidad:** Población y densidad media-baja. Índice de envejecimiento intermedio. Porcentaje de extranjeros intermedio.
    *   **Economía:** Renta neta media por persona y hogar por debajo del promedio general, mayor porcentaje de ingresos bajos (`pctingresosbajos7500_log1p`) y alto riesgo de pobreza (`pctriesgopobreza60_log1p`). Índice Gini intermedio.
    *   **Accesibilidad y Servicios:** Distancias a servicios mayores que el Clúster "Áreas Urbanas/Ricas", lo que indica una accesibilidad intermedia o menor a servicios clave.
    *   **Distribución Insular:** Concentrado en Gran Canaria y Tenerife.
    *   **Tipología:** "Áreas con menor desarrollo económico, potencialmente periféricas o con mayores desafíos socioeconómicos, y accesibilidad razonable pero no óptima a servicios."

*   **Clúster "Zonas Aisladas/Rurales de Montaña" (Anteriormente Clúster 3): Zonas Aisladas o Rurales de Montaña con Poca Renta y Alta Tasa de Dependencia (Principalmente La Gomera y El Hierro)**
    *   **Demografía y Densidad:** Baja población y muy baja densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros, alto índice de envejecimiento (`indiceenvejecimiento_log1p`) y alta tasa de dependencia (`tasadependencia`). Esto sugiere áreas rurales, envejecidas y aisladas.
    *   **Economía:** Renta neta media por hogar y persona más baja (`rentanetamediahogar_log1p`, `rentanetamediapersona`), mayor riesgo de pobreza y mayor porcentaje de ingresos bajos. Esto indica una mayor vulnerabilidad económica.
    *   **Accesibilidad y Servicios:** Distancias más altas a la mayoría de los servicios (`distminhospitalkm_log1p`, `distmincskm_log1p`, etc.), y bajo número de paradas de transporte (`nparadas500m_log1p`). Esto es coherente con su posible carácter montañoso o aislado.
    *   **Distribución Insular:** Más prevalente en islas como La Gomera y El Hierro, y algunas zonas de Fuerteventura, Lanzarote y La Palma.
    *   **Tipología:** "Regiones periféricas, rurales o de montaña, con una población envejecida, menor riqueza y mayor aislamiento."

Estos perfiles proporcionan una base para comprender las heterogeneidades territoriales en Canarias, diferenciando entre áreas urbanas prósperas, zonas rurales bien conectadas, áreas más vulnerables y regiones aisladas.

# Informe Detallado: Perfil Socioeconómico de los Clústeres K-Means (Nombres Actualizados)

Este informe presenta una descripción detallada de los perfiles socioeconómicos y geográficos de los cuatro clústeres identificados mediante el algoritmo K-Means. La interpretación se basa en las características medias de cada grupo, así como en su distribución espacial.

## Resumen de Características Medias por Clúster

### Perfil de Clústeres (Tabla Markdown con Nombres Actualizados)

| cluster_name                     | tasadependencia | nhospitales5km | pctriesgopobreza60_log1p | indicegini_log1p |
|:---------------------------------|:----------------|:---------------|:-------------------------|:-----------------|
| Zonas Rurales/Baja Densidad      | 42.3185         | 3.57258        | 3.065095                 | 3.363973         |
| Áreas Urbanas/Ricas              | 38.4128         | 14.7429        | 3.429994                 | 3.464434         |
| Áreas Vulnerables/Menos Pobladas | 51.0604         | 2.64835        | 3.284050                 | 3.435216         |
| Zonas Aisladas/Rurales de Montaña| 48.6832         | 0.52           | 3.125791                 | 3.385750         |

## Selección del Número Óptimo de Clústeres (K)

El método del codo y el coeficiente de silueta se utilizaron para determinar el número óptimo de clústeres. Aunque el método del codo no mostró un punto de inflexión muy pronunciado, el coeficiente de silueta, que mide cuán similar es un objeto a su propio clúster en comparación con otros clústeres, indicó que **K=4** fue una opción razonable, mostrando un coeficiente de **0.1872**. Si bien este valor no es extremadamente alto, es el que mejor equilibrio ofrece en la separación de los clústeres en el rango explorado. Un coeficiente de silueta cercano a 0 indica que los clústeres no están claramente separados, lo cual es común en datos complejos como los socioeconómicos, donde las transiciones entre tipologías territoriales pueden ser graduales. Esta elección de K=4 se alinea con la hipótesis de que existen alrededor de cuatro tipologías territoriales principales: zonas rurales, urbanas de alta densidad, áreas de vulnerabilidad socioeconómica y regiones más aisladas.

## Perfil Detallado de Cada Clúster (Nombres Actualizados)

### Clúster "Zonas Rurales/Baja Densidad": Zonas Rurales y de Baja Densidad con Buena Accesibilidad (Principalmente Tenerife)

*   **Demografía y Densidad:** Baja población y densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros y nacidos en el extranjero, bajo índice de envejecimiento y alta tasa de dependencia. Esto sugiere poblaciones más pequeñas, posiblemente rurales o envejecidas.
*   **Economía:** Renta neta media por hogar y persona (`rentanetamediahogar_log1p`, `rentanetamediapersona`) en rangos intermedios. Bajo riesgo de pobreza.
*   **Accesibilidad y Servicios:** Valores bajos en distancias a servicios esenciales (`distminhospitalkm_log1p`, `distmincskm_log1p`, `distminfarmaciakm_log1p`, `distmincolegiokm_log1p`). Esto indica buena accesibilidad a servicios a pesar de la baja densidad, posiblemente por la proximidad de núcleos urbanos o una infraestructura de servicios eficiente.
*   **Distribución Insular:** Predominante en Tenerife, con presencia significativa en Gran Canaria.
*   **Tipología:** "Zona rural o semi-urbana, bien conectada, con una población relativamente estable y menos diversidad demográfica."

### Clúster "Áreas Urbanas/Ricas": Áreas Urbanas y Densas con Mayor Población y Riqueza (Repartido entre islas principales)

*   **Demografía y Densidad:** Alta población (`poblaciontotal_log1p`) y la mayor densidad (`densidad_calculada_log1p`). Mayor porcentaje de extranjeros y nacidos en el extranjero. Tasa de dependencia más baja. Esto indica áreas urbanas, densamente pobladas y dinámicas.
*   **Economía:** La renta neta media por hogar y persona más alta (`rentanetamediahogar_log1p`, `rentanetamediapersona`), bajo riesgo de pobreza y bajo porcentaje de ingresos bajos. Esto sugiere zonas con mayor riqueza y oportunidades económicas.
*   **Accesibilidad y Servicios:** Distancias a servicios bajas (`distminhospitalkm_log1p`, etc.) y alto número de paradas de transporte público (`nparadas500m_log1p`). Esto confirma su carácter urbano y bien servido.
*   **Distribución Insular:** Presente en todas las islas principales (Tenerife, Gran Canaria, Lanzarote).
*   **Tipología:** "Núcleos urbanos principales, centros económicos y residenciales, con una población diversa y altos niveles de vida."

### Clúster "Áreas Vulnerables/Menos Pobladas": Zonas con Menor Población, Mayor Vulnerabilidad y Accesibilidad Intermedia (Principalmente Gran Canaria y Tenerife)

*   **Demografía y Densidad:** Población y densidad media-baja. Índice de envejecimiento intermedio. Porcentaje de extranjeros intermedio.
*   **Economía:** Renta neta media por persona y hogar por debajo del promedio general, mayor porcentaje de ingresos bajos (`pctingresosbajos7500_log1p`) y alto riesgo de pobreza (`pctriesgopobreza60_log1p`). Índice Gini intermedio.
*   **Accesibilidad y Servicios:** Distancias a servicios mayores que el Clúster "Áreas Urbanas/Ricas", lo que indica una accesibilidad intermedia o menor a servicios clave.
*   **Distribución Insular:** Concentrado en Gran Canaria y Tenerife.
*   **Tipología:** "Áreas con menor desarrollo económico, potencialmente periféricas o con mayores desafíos socioeconómicos, y accesibilidad razonable pero no óptima a servicios."

### Clúster "Zonas Aisladas/Rurales de Montaña": Zonas Aisladas o Rurales de Montaña con Poca Renta y Alta Tasa de Dependencia (Principalmente La Gomera y El Hierro)

*   **Demografía y Densidad:** Baja población y muy baja densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros, alto índice de envejecimiento (`indiceenvejecimiento_log1p`) y alta tasa de dependencia (`tasadependencia`). Esto sugiere áreas rurales, envejecidas y aisladas.
*   **Economía:** Renta neta media por hogar y persona más baja (`rentanetamediahogar_log1p`, `rentanetamediapersona`), mayor riesgo de pobreza y mayor porcentaje de ingresos bajos. Esto indica una mayor vulnerabilidad económica.
*   **Accesibilidad y Servicios:** Distancias más altas a la mayoría de los servicios (`distminhospitalkm_log1p`, `distmincskm_log1p`, etc.), y bajo número de paradas de transporte (`nparadas500m_log1p`). Esto es coherente con su posible carácter montañoso o aislado.
*   **Distribución Insular:** Más prevalente en islas como La Gomera y El Hierro, y algunas zonas de Fuerteventura, Lanzarote y La Palma.
*   **Tipología:** "Regiones periféricas, rurales o de montaña, con una población envejecida, menor riqueza y mayor aislamiento."

## Conclusión

Estos perfiles proporcionan una base para comprender las heterogeneidades territoriales en Canarias, diferenciando entre áreas urbanas prósperas, zonas rurales bien conectadas, áreas más vulnerables y regiones aisladas. La caracterización de estos clústeres es fundamental para informar decisiones de planificación territorial y desarrollo socioeconómico.

In [ ]:
print("--- Recalculando el perfil de las características medias para cada clúster ---")

# Asegurarse de que merged_df está cargado y tiene 'cluster_label'
if 'merged_df' not in locals():
    raise RuntimeError("'merged_df' no está disponible. Asegúrate de ejecutar las celdas de carga de datos.")

if 'cluster_label' not in merged_df.columns:
    # Attempt to recreate cluster_label if it's missing (as done in 'aa0d1e66')
    print("Advertencia: 'cluster_label' no encontrado en 'merged_df'. Intentando recrear clústeres K-Means.")
    if 'X_scaled_final_df' in locals() and not X_scaled_final_df.empty:
        if 'optimal_k' not in locals(): optimal_k = 4 # Default optimal_k
        kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
        merged_df['cluster_label'] = kmeans.fit_predict(X_scaled_final_df)
        print(f"'cluster_label' recreado con {optimal_k} clusters.")
    else:
        raise RuntimeError("No se pudo recrear 'cluster_label' porque 'X_scaled_final_df' no está disponible o está vacío.")

# Ensure clustering_features list is available. It is defined in 'd6d7037e'.
# Recreate if not found to ensure robustness.
if 'clustering_features' not in locals():
    print("Advertencia: 'clustering_features' no definida. Recreando a partir de la lógica original.")
    # This part duplicates logic from d6d7037e - base_clustering_features, highly_skewed_cols
    # For simplicity, assuming X_scaled_final_df's columns are representative of selected features.

    # Robust recreation of highly_skewed_cols if not found (for log1p prioritization)
    if 'highly_skewed_cols' not in locals():
        print("Warning: 'highly_skewed_cols' not defined. Attempting to recreate.")
        exclude_cols_from_skew = ['cusec', 'objectid', 'objectid1']
        numerical_cols_temp = merged_df.select_dtypes(include=np.number).columns.tolist()
        numerical_cols_temp = [col for col in numerical_cols_temp if col not in exclude_cols_from_skew]
        existing_numerical_cols = [col for col in numerical_cols_temp if col in merged_df.columns]

        if not existing_numerical_cols:
            skew_values_temp = pd.Series(dtype='float64')
        else:
            skew_values_temp = merged_df[existing_numerical_cols].apply(lambda x: skew(x.dropna()))

        skewness_threshold = 1.0
        highly_skewed_cols = skew_values_temp[skew_values_temp > skewness_threshold].index.tolist()
        print("'highly_skewed_cols' recreated for consistency.")

    # Define a base list of meaningful features for clustering territorial typologies.
    base_clustering_features = [
        'poblaciontotal', 'densidad_calculada', 'rentanetamediahogar', 'rentanetamediapersona',
        'pctextranjeros', 'pctnacidosextranjero', 'indiceenvejecimiento', 'tasadependencia',
        'pensionmedia', 'pctingresosbajos7500', 'pctriesgopobreza60', 'pctingresosaltos200',
        'indicegini', 'spersonas',
        'distminhospitalkm', 'distmincskm', 'distminfarmaciakm', 'distmincolegiokm',
        'distminparadakm', 'nparadas500m', 'nhospitales5km',
        'estimated_persons_per_household',
        'relative_island_density', 'relative_island_household_wealth',
        'healthcare_offer_demand_ratio',
        'demographic_service_pressure'
    ]

    clustering_features = []
    for col in base_clustering_features:
        log1p_col_name = col + '_log1p'
        if log1p_col_name in merged_df.columns and col in highly_skewed_cols:
            clustering_features.append(log1p_col_name)
        elif col in merged_df.columns:
            clustering_features.append(col)
        else:
            print(f"Advertencia: La característica base '{col}' no se encontró en 'merged_df' y será omitida de clustering_features.")
    clustering_features = list(set([f for f in clustering_features if f in merged_df.select_dtypes(include=np.number).columns]))
    print("'clustering_features' list recreated.")

if not clustering_features:
    raise RuntimeError("La lista 'clustering_features' está vacía. No se puede generar el perfil del clúster.")

# Filter clustering_features to include only those columns actually present in merged_df
existing_clustering_features = [col for col in clustering_features if col in merged_df.columns]

# Seleccionar las características de clustering del merged_df y la etiqueta del clúster
cluster_profile_data = merged_df[existing_clustering_features + ['cluster_label']].copy()

# Calcular la media de cada característica por clúster
cluster_summary = cluster_profile_data.groupby('cluster_label')[existing_clustering_features].mean()

print("Perfil de características medias por clúster (basado en características de clustering):")
display(cluster_summary)

print("\n--- Distribución de Clústeres por Isla (Conteo) ---")
cluster_distribution_by_isla = merged_df.groupby(['isla', 'cluster_label']).size().unstack(fill_value=0)
display(cluster_distribution_by_isla)

print("\nInterpretación:")
print("Para interpretar cada clúster, compare los valores medios de las características con el promedio general de los datos. Por ejemplo:")
print("- **Población/Densidad:** Clústeres con valores altos podrían representar zonas urbanas/densas.")
print("- **Renta:** Clústeres con alta 'rentanetamediahogar_log1p' o 'rentanetamediapersona_log1p' indican mayor riqueza.")
print("- **Accesibilidad/Distancias:** Valores bajos en 'distminhospitalkm' o altos en 'nparadas500m' sugieren buena accesibilidad.")
print("- **Demografía:** 'pctextranjeros', 'indiceenvejecimiento' y 'tasadependencia' revelan el tipo de población.")
print("Analice también la distribución de los clústeres entre las diferentes islas para identificar patrones geográficos.")

### Interpretación del Perfil de Clústeres y Selección de K

Los K-Means clusters identificados (K=4) representan diferentes tipologías territoriales dentro de las Islas Canarias, caracterizadas por sus perfiles socioeconómicos y geográficos. A continuación, se detalla la interpretación de cada clúster basada en las medias de las características.

#### Selección del Número Óptimo de Clústeres (K)

El método del codo y el coeficiente de silueta se utilizaron para determinar el número óptimo de clústeres. Aunque el método del codo no mostró un punto de inflexión muy pronunciado, el coeficiente de silueta, que mide cuán similar es un objeto a su propio clúster en comparación con otros clústeres, indicó que **K=4** fue una opción razonable, mostrando un coeficiente de **0.1651**. Si bien este valor no es extremadamente alto, es el que mejor equilibrio ofrece en la separación de los clústeres en el rango explorado (ver gráfica del coeficiente de silueta en la celda anterior). Un coeficiente de silueta cercano a 0 indica que los clústeres no están claramente separados, lo cual es común en datos complejos como los socioeconómicos, donde las transiciones entre tipologías territoriales pueden ser graduales. Esta elección de K=4 se alinea con la hipótesis de que existen alrededor de cuatro tipologías territoriales principales: zonas rurales, urbanas de alta densidad, áreas de vulnerabilidad socioeconómica y regiones más aisladas.

#### Perfil de Cada Clúster

*   **Clúster 0: Zonas Rurales y de Baja Densidad con Buena Accesibilidad (Principalmente Tenerife)**
    *   **Demografía y Densidad:** Baja población y densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros y nacidos en el extranjero, bajo índice de envejecimiento y alta tasa de dependencia. Esto sugiere poblaciones más pequeñas, posiblemente rurales o envejecidas.
    *   **Economía:** Renta neta media por hogar y persona (`rentanetamediahogar_log1p`, `rentanetamediapersona`) en rangos intermedios. Bajo riesgo de pobreza.
    *   **Accesibilidad y Servicios:** Valores bajos en distancias a servicios esenciales (`distminhospitalkm_log1p`, `distmincskm_log1p`, `distminfarmaciakm_log1p`, `distmincolegiokm_log1p`). Esto indica buena accesibilidad a servicios a pesar de la baja densidad, posiblemente por la proximidad de núcleos urbanos o una infraestructura de servicios eficiente.
    *   **Distribución Insular:** Predominante en Tenerife, con presencia significativa en Gran Canaria.
    *   **Tipología:** "Zona rural o semi-urbana, bien conectada, con una población relativamente estable y menos diversidad demográfica."

*   **Clúster 1: Áreas Urbanas y Densas con Mayor Población y Riqueza (Repartido entre islas principales)**
    *   **Demografía y Densidad:** Alta población (`poblaciontotal_log1p`) y la mayor densidad (`densidad_calculada_log1p`). Mayor porcentaje de extranjeros y nacidos en el extranjero. Tasa de dependencia más baja. Esto indica áreas urbanas, densamente pobladas y dinámicas.
    *   **Economía:** La renta neta media por hogar y persona más alta (`rentanetamediahogar_log1p`, `rentanetamediapersona`), bajo riesgo de pobreza y bajo porcentaje de ingresos bajos. Esto sugiere zonas con mayor riqueza y oportunidades económicas.
    *   **Accesibilidad y Servicios:** Distancias a servicios bajas (`distminhospitalkm_log1p`, etc.) y alto número de paradas de transporte público (`nparadas500m_log1p`). Esto confirma su carácter urbano y bien servido.
    *   **Distribución Insular:** Presente en todas las islas principales (Tenerife, Gran Canaria, Lanzarote).
    *   **Tipología:** "Núcleos urbanos principales, centros económicos y residenciales, con una población diversa y altos niveles de vida."

*   **Clúster 2: Zonas con Menor Población, Mayor Vulnerabilidad y Accesibilidad Intermedia (Principalmente Gran Canaria y Tenerife)**
    *   **Demografía y Densidad:** Población y densidad media-baja. Índice de envejecimiento intermedio. Porcentaje de extranjeros intermedio.
    *   **Economía:** Renta neta media por persona y hogar por debajo del promedio general, mayor porcentaje de ingresos bajos (`pctingresosbajos7500_log1p`) y alto riesgo de pobreza (`pctriesgopobreza60_log1p`). Índice Gini intermedio.
    *   **Accesibilidad y Servicios:** Distancias a servicios mayores que el Clúster 1, lo que indica una accesibilidad intermedia o menor a servicios clave.
    *   **Distribución Insular:** Concentrado en Gran Canaria y Tenerife.
    *   **Tipología:** "Áreas con menor desarrollo económico, potencialmente periféricas o con mayores desafíos socioeconómicos, y accesibilidad razonable pero no óptima a servicios."

*   **Clúster 3: Zonas Aisladas o Rurales de Montaña con Poca Renta y Alta Tasa de Dependencia (Principalmente La Gomera y El Hierro)**
    *   **Demografía y Densidad:** Baja población y muy baja densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros, alto índice de envejecimiento (`indiceenvejecimiento_log1p`) y alta tasa de dependencia (`tasadependencia`). Esto sugiere áreas rurales, envejecidas y aisladas.
    *   **Economía:** Renta neta media por hogar y persona más baja (`rentanetamediahogar_log1p`, `rentanetamediapersona`), mayor riesgo de pobreza y mayor porcentaje de ingresos bajos. Esto indica una mayor vulnerabilidad económica.
    *   **Accesibilidad y Servicios:** Distancias más altas a la mayoría de los servicios (`distminhospitalkm_log1p`, `distmincskm_log1p`, etc.), y bajo número de paradas de transporte (`nparadas500m_log1p`). Esto es coherente con su posible carácter montañoso o aislado.
    *   **Distribución Insular:** Más prevalente en islas como La Gomera y El Hierro, y algunas zonas de Fuerteventura, Lanzarote y La Palma.
    *   **Tipología:** "Regiones periféricas, rurales o de montaña, con una población envejecida, menor riqueza y mayor aislamiento."

Estos perfiles proporcionan una base para comprender las heterogeneidades territoriales en Canarias, diferenciando entre áreas urbanas prósperas, zonas rurales bien conectadas, áreas más vulnerables y regiones aisladas.

# Informe Detallado: Perfil Socioeconómico de los Clústeres K-Means

Este informe presenta una descripción detallada de los perfiles socioeconómicos y geográficos de los cuatro clústeres identificados mediante el algoritmo K-Means. La interpretación se basa en las características medias de cada grupo, así como en su distribución espacial.

## Resumen de Características Medias por Clúster

### Perfil de Clústeres (Tabla Markdown)

| cluster_label | tasadependencia | nhospitales5km | pctriesgopobreza60_log1p | indicegini_log1p |
|:--------------|:----------------|:---------------|:-------------------------|:-----------------|
| 0             | 42.3185         | 3.57258        | 3.065095                 | 3.363973         |
| 1             | 38.4128         | 14.7429        | 3.429994                 | 3.464434         |
| 2             | 51.0604         | 2.64835        | 3.284050                 | 3.435216         |
| 3             | 48.6832         | 0.52           | 3.125791                 | 3.385750         |

## Selección del Número Óptimo de Clústeres (K)

El método del codo y el coeficiente de silueta se utilizaron para determinar el número óptimo de clústeres. Aunque el método del codo no mostró un punto de inflexión muy pronunciado, el coeficiente de silueta, que mide cuán similar es un objeto a su propio clúster en comparación con otros clústeres, indicó que **K=4** fue una opción razonable, mostrando un coeficiente de **0.1872**. Si bien este valor no es extremadamente alto, es el que mejor equilibrio ofrece en la separación de los clústeres en el rango explorado. Un coeficiente de silueta cercano a 0 indica que los clústeres no están claramente separados, lo cual es común en datos complejos como los socioeconómicos, donde las transiciones entre tipologías territoriales pueden ser graduales. Esta elección de K=4 se alinea con la hipótesis de que existen alrededor de cuatro tipologías territoriales principales: zonas rurales, urbanas de alta densidad, áreas de vulnerabilidad socioeconómica y regiones más aisladas.

## Perfil Detallado de Cada Clúster

### Clúster 0: Zonas Rurales y de Baja Densidad con Buena Accesibilidad (Principalmente Tenerife)

*   **Demografía y Densidad:** Baja población y densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros y nacidos en el extranjero, bajo índice de envejecimiento y alta tasa de dependencia. Esto sugiere poblaciones más pequeñas, posiblemente rurales o envejecidas.
*   **Economía:** Renta neta media por hogar y persona (`rentanetamediahogar_log1p`, `rentanetamediapersona`) en rangos intermedios. Bajo riesgo de pobreza.
*   **Accesibilidad y Servicios:** Valores bajos en distancias a servicios esenciales (`distminhospitalkm_log1p`, `distmincskm_log1p`, `distminfarmaciakm_log1p`, `distmincolegiokm_log1p`). Esto indica buena accesibilidad a servicios a pesar de la baja densidad, posiblemente por la proximidad de núcleos urbanos o una infraestructura de servicios eficiente.
*   **Distribución Insular:** Predominante en Tenerife, con presencia significativa en Gran Canaria.
*   **Tipología:** "Zona rural o semi-urbana, bien conectada, con una población relativamente estable y menos diversidad demográfica."

### Clúster 1: Áreas Urbanas y Densas con Mayor Población y Riqueza (Repartido entre islas principales)

*   **Demografía y Densidad:** Alta población (`poblaciontotal_log1p`) y la mayor densidad (`densidad_calculada_log1p`). Mayor porcentaje de extranjeros y nacidos en el extranjero. Tasa de dependencia más baja. Esto indica áreas urbanas, densamente pobladas y dinámicas.
*   **Economía:** La renta neta media por hogar y persona más alta (`rentanetamediahogar_log1p`, `rentanetamediapersona`), bajo riesgo de pobreza y bajo porcentaje de ingresos bajos. Esto sugiere zonas con mayor riqueza y oportunidades económicas.
*   **Accesibilidad y Servicios:** Distancias a servicios bajas (`distminhospitalkm_log1p`, etc.) y alto número de paradas de transporte público (`nparadas500m_log1p`). Esto confirma su carácter urbano y bien servido.
*   **Distribución Insular:** Presente en todas las islas principales (Tenerife, Gran Canaria, Lanzarote).
*   **Tipología:** "Núcleos urbanos principales, centros económicos y residenciales, con una población diversa y altos niveles de vida."

### Clúster 2: Zonas con Menor Población, Mayor Vulnerabilidad y Accesibilidad Intermedia (Principalmente Gran Canaria y Tenerife)

*   **Demografía y Densidad:** Población y densidad media-baja. Índice de envejecimiento intermedio. Porcentaje de extranjeros intermedio.
*   **Economía:** Renta neta media por persona y hogar por debajo del promedio general, mayor porcentaje de ingresos bajos (`pctingresosbajos7500_log1p`) y alto riesgo de pobreza (`pctriesgopobreza60_log1p`). Índice Gini intermedio.
*   **Accesibilidad y Servicios:** Distancias a servicios mayores que el Clúster 1, lo que indica una accesibilidad intermedia o menor a servicios clave.
*   **Distribución Insular:** Concentrado en Gran Canaria y Tenerife.
*   **Tipología:** "Áreas con menor desarrollo económico, potencialmente periféricas o con mayores desafíos socioeconómicos, y accesibilidad razonable pero no óptima a servicios."

### Clúster 3: Zonas Aisladas o Rurales de Montaña con Poca Renta y Alta Tasa de Dependencia (Principalmente La Gomera y El Hierro)

*   **Demografía y Densidad:** Baja población y muy baja densidad (`poblaciontotal_log1p`, `densidad_calculada_log1p`), bajo porcentaje de extranjeros, alto índice de envejecimiento (`indiceenvejecimiento_log1p`) y alta tasa de dependencia (`tasadependencia`). Esto sugiere áreas rurales, envejecidas y aisladas.
*   **Economía:** Renta neta media por hogar y persona más baja (`rentanetamediahogar_log1p`, `rentanetamediapersona`), mayor riesgo de pobreza y mayor porcentaje de ingresos bajos. Esto indica una mayor vulnerabilidad económica.
*   **Accesibilidad y Servicios:** Distancias más altas a la mayoría de los servicios (`distminhospitalkm_log1p`, `distmincskm_log1p`, etc.), y bajo número de paradas de transporte (`nparadas500m_log1p`). Esto es coherente con su posible carácter montañoso o aislado.
*   **Distribución Insular:** Más prevalente en islas como La Gomera y El Hierro, y algunas zonas de Fuerteventura, Lanzarote y La Palma.
*   **Tipología:** "Regiones periféricas, rurales o de montaña, con una población envejecida, menor riqueza y mayor aislamiento."

## Conclusión

Estos perfiles proporcionan una base para comprender las heterogeneidades territoriales en Canarias, diferenciando entre áreas urbanas prósperas, zonas rurales bien conectadas, áreas más vulnerables y regiones aisladas. La caracterización de estos clústeres es fundamental para informar decisiones de planificación territorial y desarrollo socioeconómico.

### Radar de Perfiles de Clústeres

Para visualizar los perfiles de cada clúster de una manera intuitiva y comparativa, utilizaremos un gráfico de radar. Este gráfico mostrará cómo se posiciona cada clúster en relación con las principales características que definen las tipologías territoriales. Esto es ideal para audiencias no técnicas, ya que permite ver el perfil completo de una zona de un vistazo.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import skew # Added import

print("--- Generando Radar de Perfiles de Clústeres ---")

if 'merged_df' in locals() and 'cluster_label' in merged_df.columns:
    # Define a base list of meaningful features for the radar chart.
    base_radar_features = [
        'poblaciontotal', # Tamaño
        'rentanetamediahogar', # Riqueza
        'general_accessibility_score', # Accesibilidad general
        'isolation_index_normalized', # Aislamiento (ya normalizado 0-1)
        'healthcare_offer_demand_ratio', # Salud
        'tasadependencia' # Demografía
    ]

    # --- Start of added logic to ensure _log1p features are present ---
    # Re-calculate skew values and identify highly skewed columns for features that should be transformed
    exclude_cols_from_skew = ['cusec', 'objectid', 'objectid1'] # Common exclusions
    numerical_cols = merged_df.select_dtypes(include=np.number).columns.tolist()
    numerical_cols = [col for col in numerical_cols if col not in exclude_cols_from_skew]
    skew_values = merged_df[numerical_cols].apply(lambda x: skew(x.dropna()))
    skewness_threshold = 2
    highly_skewed_cols_current = skew_values[skew_values > skewness_threshold].index.tolist()

    print(f"Debug (re-calc): Highly skewed features identified: {highly_skewed_cols_current[:5]}...")

    # For base_radar_features that are highly skewed, ensure their _log1p version exists
    for feature in base_radar_features:
        log1p_feature = feature + '_log1p'
        if feature in highly_skewed_cols_current and log1p_feature not in merged_df.columns:
            # New: Ensure feature is non-negative before applying log1p
            if (merged_df[feature] < 0).any():
                print(f"Debug (re-apply): Clamping negative values in '{feature}' to 0 before applying log1p.")
                merged_df[feature] = merged_df[feature].clip(lower=0)

            # Apply log1p directly after clamping, as it handles NaNs by propagating them.
            merged_df[log1p_feature] = np.log1p(merged_df[feature])
            print(f"Debug (re-apply): Transformed '{feature}' to '{log1p_feature}' for radar chart.")
    # --- End of added logic ---

    # Convert highly_skewed_cols_current to a list for robust checking
    highly_skewed_features_list = highly_skewed_cols_current # Updated to use the re-calculated list
    print(f"Debug: Highly skewed features list: {highly_skewed_features_list}")
    print(f"Debug: merged_df columns example: {merged_df.columns.tolist()[:10]}")

    # Dynamically select features, preferring log1p versions if available and original was highly skewed
    radar_features = []
    for feature in base_radar_features:
        log1p_feature = feature + '_log1p'
        print(f"Debug: Checking feature '{feature}'")
        is_skewed = feature in highly_skewed_features_list
        has_log1p_version = log1p_feature in merged_df.columns
        print(f"Debug: Is '{feature}' in highly_skewed_features_list? {is_skewed}")
        print(f"Debug: Is '{log1p_feature}' in merged_df.columns? {has_log1p_version}")

        # Prioritize log1p version if it exists AND the original feature was identified as highly skewed
        if is_skewed and has_log1p_version:
            radar_features.append(log1p_feature)
            print(f"Debug: Appended transformed feature: {log1p_feature}")
        # Otherwise, use the original feature if it exists
        elif feature in merged_df.columns:
            radar_features.append(feature)
            print(f"Debug: Appended original feature: {feature}")
        else:
            print(f"Advertencia: La característica '{feature}' (ni su versión log1p) no se encontró en 'merged_df' y será omitida.")

    if not radar_features:
        print("Error: No se encontraron características válidas para el gráfico de radar.")
    else:
        print(f"Características seleccionadas para el radar chart: {radar_features}")

        # Calcular la media de cada característica por clúster
        cluster_means = merged_df.groupby('cluster_label')[radar_features].mean()

        # Normalizar los valores para el radar chart (MinMax entre todos los clusters para cada característica)
        # Esto asegura que los valores estén en una escala comparable [0, 1] o similar para la visualización.
        normalized_cluster_means = cluster_means.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)

        # Número de variables y ángulos para el radar chart
        num_vars = len(radar_features)
        angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

        # El gráfico de radar debe cerrarse, así que añadimos el primer ángulo al final
        if num_vars > 0:
            # Ensure the DataFrame has enough columns for concat. If num_vars is 0, this would cause an error.
            # This check is technically redundant due to 'if not radar_features' above, but good for robustness.
            if len(normalized_cluster_means.columns) > 0:
                normalized_cluster_means = pd.concat([normalized_cluster_means, normalized_cluster_means.iloc[:, :1]], axis=1)
                angles = angles + angles[:1]
            else:
                print("Advertencia: No hay columnas para concatenar en normalized_cluster_means. No se cerrará el radar.")

        # Crear el radar chart
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

        # Colores para cada clúster
        colors = plt.get_cmap('tab10', len(normalized_cluster_means.index))

        for i, (cluster_id, row) in enumerate(normalized_cluster_means.iterrows()):
            ax.plot(angles, row.values, color=colors(i), linewidth=2, linestyle='solid', label=f'Clúster {cluster_id}')
            ax.fill(angles, row.values, color=colors(i), alpha=0.25)

        # Configuración del gráfico
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        ax.set_rlabel_position(0) # Posición de las etiquetas de los radios
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(radar_features, fontsize=12)
        ax.tick_params(axis='y', labelsize=10) # Tamaño de las etiquetas de la escala radial
        ax.set_title('Radar de Perfiles de Clústeres', va='bottom', fontsize=16)
        ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        ax.grid(True)
        plt.show()

else:
    print("Error: 'merged_df' o 'cluster_label' no están disponibles. Asegúrate de que las celdas anteriores se hayan ejecutado correctamente.")

Como análisis exploratorio previo al modelado, se aplica K-Means sobre las variables normalizadas para generar una hipótesis de partida sobre el número de tipologías territoriales presentes en los datos brutos. El valor K obtenido (K=4) se toma como referencia orientativa para el clustering aplicado posteriormente sobre el espacio latente del autoencoder. Si el autoencoder descubre la misma estructura, confirma que los datos brutos contienen información suficiente. Si descubre una estructura diferente, ese hallazgo constituye en sí mismo un resultado de la comparativa y responde directamente a PG1 y PG2.